# Individual-return and managed-portfolio evaluation

Research code from my MSc thesis, University of Copenhagen, 2024.

Original research cells are preserved; saved outputs are cleared. See the repository README and docs/source_notes.md for inputs and execution context.


**Economic performance evaluation - Equally weighted portfolios**

In [ ]:
import pandas as pd
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')


# Define the HDF5 file path
hdf5_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/new_results_CA_0/new_scores.h5')

# Open the HDF5 file to list all keys and load predictions
with pd.HDFStore(hdf5_path, mode='r') as store:
    keys = store.keys()
    print("Available keys in the HDF5 file:")
    for key in keys:
        print(key)

    # Dictionary to store loaded predictions grouped by n_factors
    predictions_by_n_factors = {}

    # Filter keys for combined predictions by n_factors
    for key in keys:
        if key.startswith('/predictions/') and key.count('/') == 2:  # This ensures we're looking for keys like 'predictions/n_factors'
            n_factors = key.split('/')[-1]  # Extract the n_factors value from the key

            # Load the predictions DataFrame
            predictions_df = store.get(key)

            # Add the predictions DataFrame to the dictionary for this n_factors
            predictions_by_n_factors[n_factors] = predictions_df

            print(f"Loaded combined predictions for n_factors: {n_factors}, key: {key}")
            print(predictions_df.head())  # Display the first few rows

# Now predictions_by_n_factors is a dictionary where keys are n_factors and values are the combined DataFrames


In [ ]:
!pip install tqdm

In [ ]:
!pip install backtrader

In [ ]:
# List all available n_factors keys
available_n_factors = list(predictions_by_n_factors.keys())
print("Available n_factors keys:")
print(available_n_factors)


In [ ]:
# Access the DataFrame for a specific n_factors value, e.g., '3'
n_factors_value = '3'  # Replace this with the n_factors value you want to access
if n_factors_value in predictions_by_n_factors:
    df = predictions_by_n_factors[n_factors_value]
    print(f"DataFrame for n_factors = {n_factors_value}:")
    print(df.head())
else:
    print(f"n_factors value '{n_factors_value}' not found in predictions_by_n_factors")


In [ ]:
import pandas as pd

# Assuming df is already defined and has columns: 'date', 'permno', 'actual returns', 'predicted returns'
df.rename(columns={'actual predictions': 'actual_returns', 'predicted returns': 'predicted_returns'}, inplace=True)
df['datetime'] = pd.to_datetime(df['date'])
df.set_index('datetime', inplace=True)
df.sort_index(inplace=True)


In [ ]:
print(df)

In [ ]:
print(df.columns)

In [ ]:
!pip install pandas tqdm


**Using ANNUAL cumulative returns instead of monthly returns (gives more realistic results**

In [ ]:
import pandas as pd
from pathlib import Path
from google.colab import drive
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

# Mount Google Drive
drive.mount('/content/drive')

# Load the original dataset
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/beta_input_CRSP.parquet'
original_df = pd.read_parquet(file_path)

# Convert date columns to the same format
original_df['date'] = pd.to_datetime(original_df['date'])

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/REVISED_results_CA_0')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Define the HDF5 file paths for different models
model_paths = {
    # 'CA_0': '/content/drive/My Drive/Tidy Finance ML dataset/REVISED_results_CA_0/results/asset_pricing/predictions.h5',
    # 'CA_1': '/content/drive/My Drive/Tidy Finance ML dataset/REVISED_results_CA_1/results/asset_pricing/predictions.h5',
    'CA_2': '/content/drive/My Drive/Tidy Finance ML dataset/REVISED_results_CA_2/results/asset_pricing/predictions.h5'
    # 'CA_3': '/content/drive/My Drive/Tidy Finance ML dataset/REVISED_results_CA_3/results/asset_pricing/predictions.h5',
}

# Function to load predictions for a given model
def load_predictions(hdf5_path):
    predictions_by_n_factors = {}
    with pd.HDFStore(hdf5_path, mode='r') as store:
        keys = store.keys()
        for key in keys:
            if key.startswith('/predictions/') and key.count('/') == 2:  # Ensure we're looking for keys like 'predictions/n_factors'
                n_factors = key.split('/')[-1]  # Extract the n_factors value from the key
                predictions_df = store.get(key)  # Load the predictions DataFrame
                predictions_df['date'] = pd.to_datetime(predictions_df['date'])  # Convert date to datetime
                predictions_df['permno'] = predictions_df['permno'].astype(str).str.extract('(\d+)').astype(int)  # Extract integer part and convert to int
                predictions_by_n_factors[n_factors] = predictions_df
    return predictions_by_n_factors

# Load predictions for all models
all_predictions = {model: load_predictions(path) for model, path in model_paths.items()}

# Function to calculate value-weighted decile portfolios
def calculate_deciles(predicted, actual, mktcap_lag):
    deciles = pd.qcut(predicted.rank(method='first'), 10, labels=False) + 1
    top_decile = actual[deciles == 10]
    bottom_decile = actual[deciles == 1]
    top_weights = mktcap_lag[deciles == 10]
    bottom_weights = mktcap_lag[deciles == 1]
    top_return = np.average(top_decile, weights=top_weights)
    bottom_return = np.average(bottom_decile, weights=bottom_weights)
    zero_net_investment_return = top_return - bottom_return
    return zero_net_investment_return

# Function to evaluate economic performance for each model
def evaluate_performance(predictions_df, original_df):
    predictions_df = predictions_df.sort_values(by='date')
    merged_df = pd.merge(predictions_df, original_df[['date', 'permno', 'mktcap_lag']], on=['date', 'permno'])
    merged_df['year'] = merged_df['date'].dt.year
    annual_returns = merged_df.groupby('year').apply(lambda x: calculate_deciles(x['predicted returns'], x['actual returns'], x['mktcap_lag']))

    performance_df = pd.DataFrame({
        'Zero_Net_Investment_Returns': annual_returns
    })
    performance_df['Cumulative_Returns'] = (1 + performance_df['Zero_Net_Investment_Returns']).cumprod() - 1

    sharpe_ratio = performance_df['Zero_Net_Investment_Returns'].mean() / performance_df['Zero_Net_Investment_Returns'].std()

    return performance_df, sharpe_ratio

# Dictionary to store Sharpe ratios
sharpe_ratios = {model: {} for model in model_paths.keys()}

# Evaluate performance for each model and factor combination and plot the results
for n_factors in tqdm(all_predictions['CA_2'].keys(), desc="Processing factors"):
    plt.figure(figsize=(10, 6))
    for model in model_paths.keys():
        predictions_df = all_predictions[model][n_factors]
        performance_df, sharpe_ratio = evaluate_performance(predictions_df, original_df)
        sharpe_ratios[model][n_factors] = sharpe_ratio
        plt.plot(performance_df.index, performance_df['Cumulative_Returns'], label=f'{model} (Sharpe: {sharpe_ratio:.2f})')
    plt.title(f'Cumulative Returns for Factor {n_factors}')
    plt.xlabel('Year')
    plt.ylabel('Cumulative Returns')
    plt.legend()
    plt.show()

# Create a DataFrame for Sharpe ratios
sharpe_df = pd.DataFrame(sharpe_ratios).T

# Convert the DataFrame to a LaTeX table
latex_table = sharpe_df.to_latex(index=True, header=True, float_format="%.2f")

# Print the LaTeX table
print(latex_table)


In [ ]:
import pandas as pd
from pathlib import Path
from google.colab import drive
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

# Mount Google Drive
drive.mount('/content/drive')

# Define the HDF5 file paths for different models
model_paths = {
    'CA_0': '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_0/new_scores.h5',
    'CA_1': '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_1/new_scores.h5',
    'CA_2': '/content/drive/My Drive/Tidy Finance ML dataset/new_results_CA_2/new_scores.h5',
    'CA_3': '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3/new_scores.h5',
}

# Function to load predictions for a given model
def load_predictions(hdf5_path):
    predictions_by_n_factors = {}
    with pd.HDFStore(hdf5_path, mode='r') as store:
        keys = store.keys()
        for key in keys:
            if key.startswith('/predictions/') and key.count('/') == 2:  # Ensure we're looking for keys like 'predictions/n_factors'
                n_factors = key.split('/')[-1]  # Extract the n_factors value from the key
                predictions_df = store.get(key)  # Load the predictions DataFrame
                predictions_by_n_factors[n_factors] = predictions_df
    return predictions_by_n_factors

# Load predictions for all models
all_predictions = {model: load_predictions(path) for model, path in model_paths.items()}

# Function to calculate decile portfolios
def calculate_deciles(predicted, actual):
    deciles = pd.qcut(predicted.rank(method='first'), 10, labels=False) + 1
    top_decile = actual[deciles == 10]
    bottom_decile = actual[deciles == 1]
    return top_decile.mean(), bottom_decile.mean()

# Function to evaluate economic performance for each model
def evaluate_performance(predictions_df):
    predictions_df = predictions_df.sort_values(by='date')
    predictions_df['year'] = predictions_df['date'].dt.year
    annual_returns = predictions_df.groupby('year').apply(lambda x: calculate_deciles(x['predicted returns'], x['actual predictions']))
    long_returns = annual_returns.apply(lambda x: x[0])
    short_returns = annual_returns.apply(lambda x: x[1])

    performance_df = pd.DataFrame({
        'Long_Returns': long_returns,
        'Short_Returns': short_returns
    })
    performance_df['Zero_Net_Investment_Returns'] = performance_df['Long_Returns'] - performance_df['Short_Returns']
    performance_df['Cumulative_Returns'] = (1 + performance_df['Zero_Net_Investment_Returns']).cumprod() - 1

    sharpe_ratio = performance_df['Zero_Net_Investment_Returns'].mean() / performance_df['Zero_Net_Investment_Returns'].std()

    return performance_df, sharpe_ratio

# Dictionary to store Sharpe ratios
sharpe_ratios = {model: {} for model in model_paths.keys()}

# Evaluate performance for each model and factor combination and plot the results
for n_factors in tqdm(all_predictions['CA_0'].keys(), desc="Processing factors"):
    plt.figure(figsize=(10, 6))
    for model in model_paths.keys():
        predictions_df = all_predictions[model][n_factors]
        performance_df, sharpe_ratio = evaluate_performance(predictions_df)
        sharpe_ratios[model][n_factors] = sharpe_ratio
        plt.plot(performance_df.index, performance_df['Cumulative_Returns'], label=f'{model} (Sharpe: {sharpe_ratio:.2f})')
    plt.title(f'Cumulative Returns for Factor {n_factors}')
    plt.xlabel('Year')
    plt.ylabel('Cumulative Returns')
    plt.legend()
    plt.show()

# Create a DataFrame for Sharpe ratios
sharpe_df = pd.DataFrame(sharpe_ratios).T

# Convert the DataFrame to a LaTeX table
latex_table = sharpe_df.to_latex(index=True, header=True, float_format="%.2f")

# Print the LaTeX table
print(latex_table)


In [ ]:
import pandas as pd
from pathlib import Path
from google.colab import drive
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

# Mount Google Drive
drive.mount('/content/drive')

# Define the HDF5 file paths for different models
model_paths = {
    'CA_0': '/content/drive/My Drive/Tidy Finance ML dataset/new_results_CA_0/new_scores.h5',
    'CA_1': '/content/drive/My Drive/Tidy Finance ML dataset/new_results_CA_1/new_scores.h5',
    'CA_2': '/content/drive/My Drive/Tidy Finance ML dataset/new_results_CA_2/new_scores.h5',
    'CA_3': '/content/drive/My Drive/Tidy Finance ML dataset/new_results_CA_3/new_scores.h5',
}

# Function to load predictions for a given model
def load_predictions(hdf5_path):
    predictions_by_n_factors = {}
    with pd.HDFStore(hdf5_path, mode='r') as store:
        keys = store.keys()
        for key in keys:
            if key.startswith('/predictions/') and key.count('/') == 2:  # Ensure we're looking for keys like 'predictions/n_factors'
                n_factors = key.split('/')[-1]  # Extract the n_factors value from the key
                predictions_df = store.get(key)  # Load the predictions DataFrame
                predictions_by_n_factors[n_factors] = predictions_df
    return predictions_by_n_factors

# Load predictions for all models
all_predictions = {model: load_predictions(path) for model, path in model_paths.items()}

# Function to calculate decile portfolios
def calculate_deciles(predicted, actual):
    deciles = pd.qcut(predicted.rank(method='first'), 10, labels=False) + 1
    top_decile = actual[deciles == 10]
    bottom_decile = actual[deciles == 1]
    return top_decile.mean(), bottom_decile.mean()

# Function to calculate additional performance metrics
def calculate_metrics(returns):
    avg_return = returns.mean()
    volatility = returns.std()
    var_95 = returns.quantile(0.05)
    turnover = (returns.diff().abs().sum()) / (2 * returns.abs().sum())
    return avg_return, volatility, var_95, turnover

# Function to evaluate economic performance for each model
def evaluate_performance(predictions_df):
    predictions_df = predictions_df.sort_values(by='date')
    predictions_df['year'] = predictions_df['date'].dt.year
    annual_returns = predictions_df.groupby('year').apply(lambda x: calculate_deciles(x['predicted returns'], x['actual predictions']))
    long_returns = annual_returns.apply(lambda x: x[0])
    short_returns = annual_returns.apply(lambda x: x[1])

    performance_df = pd.DataFrame({
        'Long_Returns': long_returns,
        'Short_Returns': short_returns
    })
    performance_df['Zero_Net_Investment_Returns'] = performance_df['Long_Returns'] - performance_df['Short_Returns']
    performance_df['Cumulative_Returns'] = (1 + performance_df['Zero_Net_Investment_Returns']).cumprod() - 1

    sharpe_ratio = performance_df['Zero_Net_Investment_Returns'].mean() / performance_df['Zero_Net_Investment_Returns'].std()

    avg_return, volatility, var_95, turnover = calculate_metrics(performance_df['Zero_Net_Investment_Returns'])

    return performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover

# Dictionaries to store metrics
sharpe_ratios = {model: {} for model in model_paths.keys()}
avg_returns = {model: {} for model in model_paths.keys()}
volatilities = {model: {} for model in model_paths.keys()}
vars_95 = {model: {} for model in model_paths.keys()}
turnovers = {model: {} for model in model_paths.keys()}

# Evaluate performance for each model and factor combination and plot the results
for n_factors in tqdm(all_predictions['CA_0'].keys(), desc="Processing factors"):
    plt.figure(figsize=(10, 6))
    for model in model_paths.keys():
        predictions_df = all_predictions[model][n_factors]
        performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover = evaluate_performance(predictions_df)
        sharpe_ratios[model][n_factors] = sharpe_ratio
        avg_returns[model][n_factors] = avg_return
        volatilities[model][n_factors] = volatility
        vars_95[model][n_factors] = var_95
        turnovers[model][n_factors] = turnover
        plt.plot(performance_df.index, performance_df['Cumulative_Returns'], label=f'{model} (Sharpe: {sharpe_ratio:.2f})')
    plt.title(f'Cumulative Returns for Factor {n_factors}')
    plt.xlabel('Year')
    plt.ylabel('Cumulative Returns')
    plt.legend()
    plt.show()

# Create DataFrames for each metric
sharpe_df = pd.DataFrame(sharpe_ratios).T
avg_returns_df = pd.DataFrame(avg_returns).T
volatilities_df = pd.DataFrame(volatilities).T
vars_95_df = pd.DataFrame(vars_95).T
turnovers_df = pd.DataFrame(turnovers).T

# Convert DataFrames to LaTeX tables
latex_sharpe = sharpe_df.to_latex(index=True, header=True, float_format="%.2f")
latex_avg_returns = avg_returns_df.to_latex(index=True, header=True, float_format="%.2f")
latex_volatilities = volatilities_df.to_latex(index=True, header=True, float_format="%.2f")
latex_vars_95 = vars_95_df.to_latex(index=True, header=True, float_format="%.2f")
latex_turnovers = turnovers_df.to_latex(index=True, header=True, float_format="%.2f")

# Print the LaTeX tables
print("Sharpe Ratios:")
print(latex_sharpe)
print("\nAverage Returns:")
print(latex_avg_returns)
print("\nVolatilities:")
print(latex_volatilities)
print("\nValue-At-Risk (95%):")
print(latex_vars_95)
print("\nTurnovers:")
print(latex_turnovers)


**VALUE WEIGHTED PORTFOLIOS**

In [ ]:
import pandas as pd
from pathlib import Path
from google.colab import drive
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

# Mount Google Drive
drive.mount('/content/drive')

# Load the original dataset
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/beta_input_CRSP.parquet'
original_df = pd.read_parquet(file_path)

# Convert date columns to the same format
original_df['date'] = pd.to_datetime(original_df['date'])

# Define the HDF5 file paths for different models
model_paths = {
    'CA_0': '/content/drive/My Drive/Tidy Finance ML dataset/new_results_CA_0/new_scores.h5',
    'CA_1': '/content/drive/My Drive/Tidy Finance ML dataset/new_results_CA_1/new_scores.h5',
    'CA_2': '/content/drive/My Drive/Tidy Finance ML dataset/new_results_CA_2/new_scores.h5',
    'CA_3': '/content/drive/My Drive/Tidy Finance ML dataset/new_results_CA_3/new_scores.h5',
}

# Function to load predictions for a given model
def load_predictions(hdf5_path):
    predictions_by_n_factors = {}
    with pd.HDFStore(hdf5_path, mode='r') as store:
        keys = store.keys()
        for key in keys:
            if key.startswith('/predictions/') and key.count('/') == 2:  # Ensure we're looking for keys like 'predictions/n_factors'
                n_factors = key.split('/')[-1]  # Extract the n_factors value from the key
                predictions_df = store.get(key)  # Load the predictions DataFrame
                predictions_df['date'] = pd.to_datetime(predictions_df['date'])  # Convert date to datetime
                predictions_df['permno'] = predictions_df['permno'].astype(int)  # Convert permno to int
                predictions_by_n_factors[n_factors] = predictions_df
    return predictions_by_n_factors

# Load predictions for all models
all_predictions = {model: load_predictions(path) for model, path in model_paths.items()}

# Function to calculate value-weighted decile portfolios
def calculate_deciles(predicted, actual, mktcap_lag):
    deciles = pd.qcut(predicted.rank(method='first'), 10, labels=False) + 1
    top_decile = actual[deciles == 10]
    bottom_decile = actual[deciles == 1]
    top_weights = mktcap_lag[deciles == 10]
    bottom_weights = mktcap_lag[deciles == 1]
    top_return = np.average(top_decile, weights=top_weights)
    bottom_return = np.average(bottom_decile, weights=bottom_weights)
    zero_net_investment_return = top_return - bottom_return
    return zero_net_investment_return

# Function to evaluate economic performance for each model
def evaluate_performance(predictions_df, original_df):
    predictions_df = predictions_df.sort_values(by='date')
    merged_df = pd.merge(predictions_df, original_df[['date', 'permno', 'mktcap_lag']], on=['date', 'permno'])
    merged_df['year'] = merged_df['date'].dt.year
    annual_returns = merged_df.groupby('year').apply(lambda x: calculate_deciles(x['predicted returns'], x['actual predictions'], x['mktcap_lag']))

    performance_df = pd.DataFrame({
        'Zero_Net_Investment_Returns': annual_returns
    })
    performance_df['Cumulative_Returns'] = (1 + performance_df['Zero_Net_Investment_Returns']).cumprod() - 1

    sharpe_ratio = performance_df['Zero_Net_Investment_Returns'].mean() / performance_df['Zero_Net_Investment_Returns'].std()

    return performance_df, sharpe_ratio

# Dictionary to store Sharpe ratios
sharpe_ratios = {model: {} for model in model_paths.keys()}

# Evaluate performance for each model and factor combination and plot the results
for n_factors in tqdm(all_predictions['CA_0'].keys(), desc="Processing factors"):
    plt.figure(figsize=(10, 6))
    for model in model_paths.keys():
        predictions_df = all_predictions[model][n_factors]
        performance_df, sharpe_ratio = evaluate_performance(predictions_df, original_df)
        sharpe_ratios[model][n_factors] = sharpe_ratio
        plt.plot(performance_df.index, performance_df['Cumulative_Returns'], label=f'{model} (Sharpe: {sharpe_ratio:.2f})')
    plt.title(f'Cumulative Returns for Factor {n_factors}')
    plt.xlabel('Year')
    plt.ylabel('Cumulative Returns')
    plt.legend()
    plt.show()

# Create a DataFrame for Sharpe ratios
sharpe_df = pd.DataFrame(sharpe_ratios).T

# Convert the DataFrame to a LaTeX table
latex_table = sharpe_df.to_latex(index=True, header=True, float_format="%.2f")

# Print the LaTeX table
print(latex_table)


In [ ]:
import pandas as pd
from pathlib import Path
from google.colab import drive
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

# Mount Google Drive
drive.mount('/content/drive')

# Load the original dataset
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/beta_input_CRSP.parquet'
original_df = pd.read_parquet(file_path)

# Convert date columns to the same format
original_df['date'] = pd.to_datetime(original_df['date'])

# Define the HDF5 file paths for different models
model_paths = {
    'CA_0': '/content/drive/My Drive/Tidy Finance ML dataset/new_results_CA_0/new_scores.h5',
    'CA_1': '/content/drive/My Drive/Tidy Finance ML dataset/new_results_CA_1/new_scores.h5',
    'CA_2': '/content/drive/My Drive/Tidy Finance ML dataset/new_results_CA_2/new_scores.h5',
    'CA_3': '/content/drive/My Drive/Tidy Finance ML dataset/new_results_CA_3/new_scores.h5',
}

# Function to load predictions for a given model
def load_predictions(hdf5_path):
    predictions_by_n_factors = {}
    with pd.HDFStore(hdf5_path, mode='r') as store:
        keys = store.keys()
        for key in keys:
            if key.startswith('/predictions/') and key.count('/') == 2:
                n_factors = key.split('/')[-1]
                predictions_df = store.get(key)
                predictions_df['date'] = pd.to_datetime(predictions_df['date'])
                predictions_df['permno'] = predictions_df['permno'].astype(int)
                predictions_by_n_factors[n_factors] = predictions_df
    return predictions_by_n_factors

# Load predictions for all models
all_predictions = {model: load_predictions(path) for model, path in model_paths.items()}

# Function to calculate value-weighted decile portfolios
def calculate_deciles(predicted, actual, mktcap_lag):
    deciles = pd.qcut(predicted.rank(method='first'), 10, labels=False) + 1
    top_decile = actual[deciles == 10]
    bottom_decile = actual[deciles == 1]
    top_weights = mktcap_lag[deciles == 10]
    bottom_weights = mktcap_lag[deciles == 1]
    top_return = np.average(top_decile, weights=top_weights)
    bottom_return = np.average(bottom_decile, weights=bottom_weights)
    zero_net_investment_return = top_return - bottom_return
    return zero_net_investment_return

# Function to calculate additional performance metrics
def calculate_metrics(returns):
    avg_return = returns.mean()
    volatility = returns.std()
    var_95 = returns.quantile(0.05)
    turnover = (returns.diff().abs().sum()) / (2 * returns.abs().sum())
    return avg_return, volatility, var_95, turnover

# Function to evaluate economic performance for each model
def evaluate_performance(predictions_df, original_df):
    predictions_df = predictions_df.sort_values(by='date')
    merged_df = pd.merge(predictions_df, original_df[['date', 'permno', 'mktcap_lag']], on=['date', 'permno'])
    merged_df['year'] = merged_df['date'].dt.year
    annual_returns = merged_df.groupby('year').apply(lambda x: calculate_deciles(x['predicted returns'], x['actual predictions'], x['mktcap_lag']))

    performance_df = pd.DataFrame({
        'Zero_Net_Investment_Returns': annual_returns
    })
    performance_df['Cumulative_Returns'] = (1 + performance_df['Zero_Net_Investment_Returns']).cumprod() - 1

    sharpe_ratio = performance_df['Zero_Net_Investment_Returns'].mean() / performance_df['Zero_Net_Investment_Returns'].std()

    avg_return, volatility, var_95, turnover = calculate_metrics(performance_df['Zero_Net_Investment_Returns'])

    return performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover

# Dictionaries to store metrics
sharpe_ratios = {model: {} for model in model_paths.keys()}
avg_returns = {model: {} for model in model_paths.keys()}
volatilities = {model: {} for model in model_paths.keys()}
vars_95 = {model: {} for model in model_paths.keys()}
turnovers = {model: {} for model in model_paths.keys()}

# Evaluate performance for each model and factor combination and plot the results
for n_factors in tqdm(all_predictions['CA_0'].keys(), desc="Processing factors"):
    plt.figure(figsize=(10, 6))
    for model in model_paths.keys():
        predictions_df = all_predictions[model][n_factors]
        performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover = evaluate_performance(predictions_df, original_df)
        sharpe_ratios[model][n_factors] = sharpe_ratio
        avg_returns[model][n_factors] = avg_return
        volatilities[model][n_factors] = volatility
        vars_95[model][n_factors] = var_95
        turnovers[model][n_factors] = turnover
        plt.plot(performance_df.index, performance_df['Cumulative_Returns'], label=f'{model} (Sharpe: {sharpe_ratio:.2f})')
    plt.title(f'Cumulative Returns for Factor {n_factors}')
    plt.xlabel('Year')
    plt.ylabel('Cumulative Returns')
    plt.legend()
    plt.show()

# Create DataFrames for each metric
sharpe_df = pd.DataFrame(sharpe_ratios).T
avg_returns_df = pd.DataFrame(avg_returns).T
volatilities_df = pd.DataFrame(volatilities).T
vars_95_df = pd.DataFrame(vars_95).T
turnovers_df = pd.DataFrame(turnovers).T

# Convert DataFrames to LaTeX tables
latex_sharpe = sharpe_df.to_latex(index=True, header=True, float_format="%.2f")
latex_avg_returns = avg_returns_df.to_latex(index=True, header=True, float_format="%.2f")
latex_volatilities = volatilities_df.to_latex(index=True, header=True, float_format="%.2f")
latex_vars_95 = vars_95_df.to_latex(index=True, header=True, float_format="%.2f")
latex_turnovers = turnovers_df.to_latex(index=True, header=True, float_format="%.2f")

# Print the LaTeX tables
print("Sharpe Ratios:")
print(latex_sharpe)
print("\nAverage Returns:")
print(latex_avg_returns)
print("\nVolatilities:")
print(latex_volatilities)
print("\nValue-At-Risk (95%):")
print(latex_vars_95)
print("\nTurnovers:")
print(latex_turnovers)


In [ ]:
import os
from pathlib import Path

# Define the path to the directory
results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_2')

# List all files in the directory
files = os.listdir(results_path)
print(f"Files in the directory: {files}")


In [ ]:
import pandas as pd
from pathlib import Path
from google.colab import drive
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

# Mount Google Drive
drive.mount('/content/drive')

# Load the original dataset
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/stock_characteristics_monthly_py.parquet'

original_df = pd.read_parquet(file_path)

# Convert date columns to the same format and reset the index
original_df = original_df.reset_index()
original_df['date'] = pd.to_datetime(original_df['date'])

# Define the base path for Google Drive
base_paths = [
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_0'),
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_1'),
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_2'),
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3')
]

# Define the results path relative to the base path
results_paths = [base_path / 'results' / 'asset_pricing' for base_path in base_paths]

# Define the HDF5 file paths for different models
model_paths = {
    'CA_0': results_paths[0] / 'all_predictions.h5',
    'CA_1': results_paths[1] / 'all_predictions.h5',
    'CA_2': results_paths[2] / 'all_predictions.h5',
    'CA_3': results_paths[3] / 'all_predictions.h5',
}

# Function to load a specific DataFrame
def load_dataframe(model, key):
    path = model_paths[model]
    if path.exists():
        with pd.HDFStore(str(path), mode='r') as store:
            if key in store.keys():
                df = store.get(key)
                return df
            else:
                print(f"Key {key} not found in {model}")
    else:
        print(f"File {path} does not exist.")
    return None

# Load predictions for all models
all_predictions = {}
for model, path in model_paths.items():
    if path.exists():
        with pd.HDFStore(str(path), mode='r') as store:
            keys = store.keys()
            all_predictions[model] = {key: store.get(key) for key in keys}

# Function to calculate value-weighted decile portfolios
def calculate_deciles(predicted, actual, mktcap_lag):
    deciles = pd.qcut(predicted.rank(method='first'), 10, labels=False) + 1
    top_decile = actual[deciles == 10]
    bottom_decile = actual[deciles == 1]
    top_weights = mktcap_lag[deciles == 10]
    bottom_weights = mktcap_lag[deciles == 1]
    top_return = np.average(top_decile, weights=top_weights)
    bottom_return = np.average(bottom_decile, weights=bottom_weights)
    zero_net_investment_return = top_return - bottom_return
    return zero_net_investment_return

# Function to calculate additional performance metrics
def calculate_metrics(returns):
    avg_return = returns.mean()
    volatility = returns.std()
    var_95 = returns.quantile(0.05)
    turnover = (returns.diff().abs().sum()) / (2 * returns.abs().sum())
    return avg_return, volatility, var_95, turnover

# Function to evaluate economic performance for each model
def evaluate_performance(predictions_df, original_df):
    predictions_df = predictions_df.sort_values(by='date')
    merged_df = pd.merge(predictions_df, original_df[['date', 'permno', 'mktcap_lag']], on=['date', 'permno'])
    merged_df['year'] = merged_df['date'].dt.year
    annual_returns = merged_df.groupby('year').apply(lambda x: calculate_deciles(x['y_pred'], x['y_true'], x['mktcap_lag']))

    performance_df = pd.DataFrame({
        'Zero_Net_Investment_Returns': annual_returns
    })
    performance_df['Cumulative_Returns'] = (1 + performance_df['Zero_Net_Investment_Returns']).cumprod() - 1

    sharpe_ratio = performance_df['Zero_Net_Investment_Returns'].mean() / performance_df['Zero_Net_Investment_Returns'].std()

    avg_return, volatility, var_95, turnover = calculate_metrics(performance_df['Zero_Net_Investment_Returns'])

    return performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover

# Dictionaries to store metrics
sharpe_ratios = {model: {} for model in model_paths.keys()}
avg_returns = {model: {} for model in model_paths.keys()}
volatilities = {model: {} for model in model_paths.keys()}
vars_95 = {model: {} for model in model_paths.keys()}
turnovers = {model: {} for model in model_paths.keys()}

# Evaluate performance for each model and factor combination and plot the results
for n_factors in tqdm(all_predictions['CA_0'].keys(), desc="Processing factors"):
    plt.figure(figsize=(10, 6))
    for model in model_paths.keys():
        predictions_df = all_predictions[model][n_factors]
        performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover = evaluate_performance(predictions_df, original_df)
        sharpe_ratios[model][n_factors] = sharpe_ratio
        avg_returns[model][n_factors] = avg_return
        volatilities[model][n_factors] = volatility
        vars_95[model][n_factors] = var_95
        turnovers[model][n_factors] = turnover
        plt.plot(performance_df.index, performance_df['Cumulative_Returns'], label=f'{model} (Sharpe: {sharpe_ratio:.2f})')
    plt.title(f'Cumulative Returns for Factor {n_factors}')
    plt.xlabel('Year')
    plt.ylabel('Cumulative Returns')
    plt.legend()
    plt.show()

# Create DataFrames for each metric
sharpe_df = pd.DataFrame(sharpe_ratios).T
avg_returns_df = pd.DataFrame(avg_returns).T
volatilities_df = pd.DataFrame(volatilities).T
vars_95_df = pd.DataFrame(vars_95).T
turnovers_df = pd.DataFrame(turnovers).T

# Convert DataFrames to LaTeX tables
latex_sharpe = sharpe_df.to_latex(index=True, header=True, float_format="%.2f")
latex_avg_returns = avg_returns_df.to_latex(index=True, header=True, float_format="%.2f")
latex_volatilities = volatilities_df.to_latex(index=True, header=True, float_format="%.2f")
latex_vars_95 = vars_95_df.to_latex(index=True, header=True, float_format="%.2f")
latex_turnovers = turnovers_df.to_latex(index=True, header=True, float_format="%.2f")

# Print the LaTeX tables
print("Sharpe Ratios:")
print(latex_sharpe)
print("\nAverage Returns:")
print(latex_avg_returns)
print("\nVolatilities:")
print(latex_volatilities)
print("\nValue-At-Risk (95%):")
print(latex_vars_95)
print("\nTurnovers:")
print(latex_turnovers)


In [ ]:
import pandas as pd
from pathlib import Path
from google.colab import drive
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

# Mount Google Drive
drive.mount('/content/drive')

# Define the base path for Google Drive
base_paths = [
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_0'),
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_1'),
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_2'),
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3')
]

# Define the results path relative to the base path
results_paths = [base_path / 'results' / 'asset_pricing' for base_path in base_paths]

# Define the HDF5 file paths for different models
model_paths = {
    'CA_0': str(results_paths[0] / 'scores.h5'),
    'CA_1': str(results_paths[1] / 'scores.h5'),
    'CA_2': str(results_paths[2] / 'scores.h5'),
    'CA_3': str(results_paths[3] / 'scores.h5'),
}

# Function to load predictions for a given model
def load_predictions(hdf5_path):
    predictions_by_n_factors = {}
    with pd.HDFStore(hdf5_path, mode='r') as store:
        keys = store.keys()
        for key in keys:
            if key.count('/') == 2:  # Ensure we're looking for keys like 'n_factors/l1_reg'
                n_factors = key.split('/')[1]  # Extract the n_factors value from the key
                predictions_df = store.get(key)  # Load the predictions DataFrame
                predictions_by_n_factors[n_factors] = predictions_df
    return predictions_by_n_factors

# Load predictions for all models
all_predictions = {model: load_predictions(path) for model, path in model_paths.items()}

# Function to calculate decile portfolios
def calculate_deciles(predicted, actual):
    deciles = pd.qcut(predicted.rank(method='first'), 10, labels=False) + 1
    top_decile = actual[deciles == 10]
    bottom_decile = actual[deciles == 1]
    return top_decile.mean(), bottom_decile.mean()

# Function to calculate additional performance metrics
def calculate_metrics(returns):
    avg_return = returns.mean()
    volatility = returns.std()
    var_95 = returns.quantile(0.05)
    turnover = (returns.diff().abs().sum()) / (2 * returns.abs().sum())
    return avg_return, volatility, var_95, turnover

# Function to evaluate economic performance for each model
def evaluate_performance(predictions_df):
    predictions_df = predictions_df.sort_values(by='date')
    predictions_df['year'] = predictions_df['date'].dt.year
    annual_returns = predictions_df.groupby('year').apply(lambda x: calculate_deciles(x['y_pred'], x['y_true']))
    long_returns = annual_returns.apply(lambda x: x[0])
    short_returns = annual_returns.apply(lambda x: x[1])

    performance_df = pd.DataFrame({
        'Long_Returns': long_returns,
        'Short_Returns': short_returns
    })
    performance_df['Zero_Net_Investment_Returns'] = performance_df['Long_Returns'] - performance_df['Short_Returns']
    performance_df['Cumulative_Returns'] = (1 + performance_df['Zero_Net_Investment_Returns']).cumprod() - 1

    sharpe_ratio = performance_df['Zero_Net_Investment_Returns'].mean() / performance_df['Zero_Net_Investment_Returns'].std()

    avg_return, volatility, var_95, turnover = calculate_metrics(performance_df['Zero_Net_Investment_Returns'])

    return performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover

# Dictionaries to store metrics
sharpe_ratios = {model: {} for model in model_paths.keys()}
avg_returns = {model: {} for model in model_paths.keys()}
volatilities = {model: {} for model in model_paths.keys()}
vars_95 = {model: {} for model in model_paths.keys()}
turnovers = {model: {} for model in model_paths.keys()}

# Evaluate performance for each model and factor combination and plot the results
for n_factors in tqdm(all_predictions['CA_0'].keys(), desc="Processing factors"):
    plt.figure(figsize=(10, 6))
    for model in model_paths.keys():
        if n_factors in all_predictions[model]:
            predictions_df = all_predictions[model][n_factors]
            performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover = evaluate_performance(predictions_df)
            sharpe_ratios[model][n_factors] = sharpe_ratio
            avg_returns[model][n_factors] = avg_return
            volatilities[model][n_factors] = volatility
            vars_95[model][n_factors] = var_95
            turnovers[model][n_factors] = turnover
            plt.plot(performance_df.index, performance_df['Cumulative_Returns'], label=f'{model} (Sharpe: {sharpe_ratio:.2f})')
    plt.title(f'Cumulative Returns for Factor {n_factors}')
    plt.xlabel('Year')
    plt.ylabel('Cumulative Returns')
    plt.legend()
    plt.show()

# Create DataFrames for each metric
sharpe_df = pd.DataFrame(sharpe_ratios).T
avg_returns_df = pd.DataFrame(avg_returns).T
volatilities_df = pd.DataFrame(volatilities).T
vars_95_df = pd.DataFrame(vars_95).T
turnovers_df = pd.DataFrame(turnovers).T

# Convert DataFrames to LaTeX tables
latex_sharpe = sharpe_df.to_latex(index=True, header=True, float_format="%.2f")
latex_avg_returns = avg_returns_df.to_latex(index=True, header=True, float_format="%.2f")
latex_volatilities = volatilities_df.to_latex(index=True, header=True, float_format="%.2f")
latex_vars_95 = vars_95_df.to_latex(index=True, header=True, float_format="%.2f")
latex_turnovers = turnovers_df.to_latex(index=True, header=True, float_format="%.2f")

# Print the LaTeX tables
print("Sharpe Ratios:")
print(latex_sharpe)
print("\nAverage Returns:")
print(latex_avg_returns)
print("\nVolatilities:")
print(latex_volatilities)
print("\nValue-At-Risk (95%):")
print(latex_vars_95)
print("\nTurnovers:")
print(latex_turnovers)


In [ ]:
import pandas as pd
from pathlib import Path

# Define the base paths for Google Drive
base_paths = [
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_0'),
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_1'),
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_2'),
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3')
]

# Initialize an empty dictionary to hold all predictions
all_predictions = {}

# Iterate over each base path
for base_path in base_paths:
    results_path = base_path / 'results' / 'asset_pricing'
    predictions_file = results_path / 'all_predictions.h5'

    # Check if the predictions file exists
    if predictions_file.exists():
        with pd.HDFStore(str(predictions_file), mode='r') as store:
            # Load all dataframes from the HDF5 file and store in the dictionary
            for key in store.keys():
                factor_l1_key = key.split('/')[-1]
                all_predictions[(base_path, factor_l1_key)] = store[key]
    else:
        print(f'Predictions file not found at: {predictions_file}')

# Example: Accessing the loaded dataframes
for key, df in all_predictions.items():
    print(f'Base Path: {key[0]}, Key: {key[1]}')
    print(df.head())  # Display the first few rows of each dataframe

# Now all_predictions contains the prediction dataframes loaded from each base path


**Annualized SR for values weighted SR portfolio (aggregated to annual)**

In [ ]:
import pandas as pd
from pathlib import Path
from google.colab   import drive
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

# Mount Google Drive
drive.mount('/content/drive')

# Load the original dataset
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/stock_characteristics_monthly_py.parquet'

original_df = pd.read_parquet(file_path)

# Rename the 'month' column to 'date'
original_df = original_df.rename(columns={'month': 'date'})

# # Ensure 'df' has a MultiIndex ['date', 'permno']
# original_df = original_df.set_index(['date', 'permno'])

# # Sort the MultiIndex
# original_df = original_df.sort_index()

# Convert date columns to the same format
original_df['date'] = pd.to_datetime(original_df['date'])



# Define the base paths for different models
base_paths = [
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_0'),
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_1'),
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_2'),
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3')
]

def load_predictions(hdf5_path):
    predictions_by_n_factors = {}
    with pd.HDFStore(hdf5_path, mode='r') as store:
        keys = store.keys()
        for key in keys:
            if key.startswith('/n_factors_'):
                n_factors = key.split('_')[2]
                predictions_df = store.get(key)
                # Reset the index to make 'date' and 'permno' regular columns
                predictions_df = predictions_df.reset_index()
                predictions_df['date'] = pd.to_datetime(predictions_df['date'])
                predictions_df['permno'] = predictions_df['permno'].astype(int)
                predictions_by_n_factors[n_factors] = predictions_df
    return predictions_by_n_factors

# Load predictions for all models
all_predictions = {}
for base_path in base_paths:
    results_path = base_path / 'results' / 'asset_pricing' / 'all_predictions.h5'
    model_name = base_path.parts[-1].split('_')[-1]
    all_predictions[model_name] = load_predictions(str(results_path))

# Function to calculate value-weighted decile portfolios
def calculate_deciles(predicted, actual, mktcap_lag):
    deciles = pd.qcut(predicted.rank(method='first'), 10, labels=False) + 1
    top_decile = actual[deciles == 10]
    bottom_decile = actual[deciles == 1]
    top_weights = mktcap_lag[deciles == 10]
    bottom_weights = mktcap_lag[deciles == 1]
    top_return = np.average(top_decile, weights=top_weights)
    bottom_return = np.average(bottom_decile, weights=bottom_weights)
    zero_net_investment_return = top_return - bottom_return
    return zero_net_investment_return

# Function to calculate additional performance metrics
def calculate_metrics(returns):
    avg_return = returns.mean()
    volatility = returns.std()
    var_95 = returns.quantile(0.05)
    turnover = (returns.diff().abs().sum()) / (2 * returns.abs().sum())
    return avg_return, volatility, var_95, turnover

# Function to evaluate economic performance for each model
def evaluate_performance(predictions_df, original_df):
    predictions_df = predictions_df.sort_values(by='date')
    merged_df = pd.merge(predictions_df, original_df[['date', 'permno', 'mktcap_lag']], on=['date', 'permno'])
    merged_df['year'] = merged_df['date'].dt.year
    annual_returns = merged_df.groupby('year').apply(lambda x: calculate_deciles(x['y_pred'], x['y_true'], x['mktcap_lag']))

    performance_df = pd.DataFrame({
        'Zero_Net_Investment_Returns': annual_returns
    })
    performance_df['Cumulative_Returns'] = (1 + performance_df['Zero_Net_Investment_Returns']).cumprod() - 1

    sharpe_ratio = performance_df['Zero_Net_Investment_Returns'].mean() / performance_df['Zero_Net_Investment_Returns'].std()

    avg_return, volatility, var_95, turnover = calculate_metrics(performance_df['Zero_Net_Investment_Returns'])

    return performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover

# Dictionaries to store metrics
sharpe_ratios = {model: {} for model in all_predictions.keys()}
avg_returns = {model: {} for model in all_predictions.keys()}
volatilities = {model: {} for model in all_predictions.keys()}
vars_95 = {model: {} for model in all_predictions.keys()}
turnovers = {model: {} for model in all_predictions.keys()}

# Evaluate performance for each model and factor combination and plot the results
for n_factors in tqdm(all_predictions['0'].keys(), desc="Processing factors"):
    plt.figure(figsize=(10, 6))
    for model in all_predictions.keys():
        predictions_df = all_predictions[model][n_factors]
        performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover = evaluate_performance(predictions_df, original_df)
        sharpe_ratios[model][n_factors] = sharpe_ratio
        avg_returns[model][n_factors] = avg_return
        volatilities[model][n_factors] = volatility
        vars_95[model][n_factors] = var_95
        turnovers[model][n_factors] = turnover
        plt.plot(performance_df.index, performance_df['Cumulative_Returns'], label=f'{model} (Sharpe: {sharpe_ratio:.2f})')
    plt.title(f'Cumulative Returns for Factor {n_factors}')
    plt.xlabel('Year')
    plt.ylabel('Cumulative Returns')
    plt.legend()
    plt.show()

# Create DataFrames for each metric
sharpe_df = pd.DataFrame(sharpe_ratios).T
avg_returns_df = pd.DataFrame(avg_returns).T
volatilities_df = pd.DataFrame(volatilities).T
vars_95_df = pd.DataFrame(vars_95).T
turnovers_df = pd.DataFrame(turnovers).T

# Convert DataFrames to LaTeX tables
latex_sharpe = sharpe_df.to_latex(index=True, header=True, float_format="%.2f")
latex_avg_returns = avg_returns_df.to_latex(index=True, header=True, float_format="%.2f")
latex_volatilities = volatilities_df.to_latex(index=True, header=True, float_format="%.2f")
latex_vars_95 = vars_95_df.to_latex(index=True, header=True, float_format="%.2f")
latex_turnovers = turnovers_df.to_latex(index=True, header=True, float_format="%.2f")

# Print the LaTeX tables
print("Sharpe Ratios:")
print(latex_sharpe)
print("\nAverage Returns:")
print(latex_avg_returns)
print("\nVolatilities:")
print(latex_volatilities)
print("\nValue-At-Risk (95%):")
print(latex_vars_95)
print("\nTurnovers:")
print(latex_turnovers)


**Annualized SR for value weighted LS portfolios time t (aggregated to monthly observations) TIME T**

In [ ]:
import pandas as pd
from pathlib import Path
from google.colab import drive
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

# Mount Google Drive
drive.mount('/content/drive')

# Load the original dataset
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/stock_characteristics_monthly_py.parquet'
original_df = pd.read_parquet(file_path)

# Rename the 'month' column to 'date'
original_df = original_df.rename(columns={'month': 'date'})

# Convert date columns to the same format
original_df['date'] = pd.to_datetime(original_df['date'])

# Define the base paths for different models
base_paths = [
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_0'),
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_1'),
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_2'),
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3')
]

def load_predictions(hdf5_path):
    predictions_by_n_factors = {}
    with pd.HDFStore(hdf5_path, mode='r') as store:
        keys = store.keys()
        for key in keys:
            if key.startswith('/n_factors_'):
                n_factors = key.split('_')[2]
                predictions_df = store.get(key)
                # Reset the index to make 'date' and 'permno' regular columns
                predictions_df = predictions_df.reset_index()
                predictions_df['date'] = pd.to_datetime(predictions_df['date'])
                predictions_df['permno'] = predictions_df['permno'].astype(int)
                predictions_by_n_factors[n_factors] = predictions_df
    return predictions_by_n_factors

# Load predictions for all models
all_predictions = {}
for base_path in base_paths:
    results_path = base_path / 'results' / 'asset_pricing' / 'all_predictions.h5'
    model_name = base_path.name  # Use the base directory name as the model name
    all_predictions[model_name] = load_predictions(str(results_path))

# Function to calculate value-weighted decile portfolios
def calculate_deciles(predicted, actual, mktcap_lag):
    deciles = pd.qcut(predicted.rank(method='first'), 10, labels=False) + 1
    top_decile = actual[deciles == 10]
    bottom_decile = actual[deciles == 1]
    top_weights = mktcap_lag[deciles == 10]
    bottom_weights = mktcap_lag[deciles == 1]
    top_return = np.average(top_decile, weights=top_weights)
    bottom_return = np.average(bottom_decile, weights=bottom_weights)
    zero_net_investment_return = top_return - bottom_return
    return zero_net_investment_return

# Function to calculate additional performance metrics
def calculate_metrics(returns):
    avg_return = returns.mean()
    volatility = returns.std()
    var_95 = returns.quantile(0.05)
    turnover = (returns.diff().abs().sum()) / (2 * returns.abs().sum())
    return avg_return, volatility, var_95, turnover

# Function to evaluate economic performance for each model
def evaluate_performance(predictions_df, original_df):
    predictions_df = predictions_df.sort_values(by='date')
    merged_df = pd.merge(predictions_df, original_df[['date', 'permno', 'mktcap_lag']], on=['date', 'permno'])
    merged_df['year'] = merged_df['date'].dt.year
    annual_returns = merged_df.groupby('year').apply(lambda x: calculate_deciles(x['y_pred'], x['y_true'], x['mktcap_lag']))

    performance_df = pd.DataFrame({
        'Zero_Net_Investment_Returns': annual_returns
    })
    performance_df['Cumulative_Returns'] = (1 + performance_df['Zero_Net_Investment_Returns']).cumprod() - 1

    sharpe_ratio = performance_df['Zero_Net_Investment_Returns'].mean() / performance_df['Zero_Net_Investment_Returns'].std()

    avg_return, volatility, var_95, turnover = calculate_metrics(performance_df['Zero_Net_Investment_Returns'])

    return performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover

# Dictionaries to store metrics
sharpe_ratios = {model: {} for model in all_predictions.keys()}
avg_returns = {model: {} for model in all_predictions.keys()}
volatilities = {model: {} for model in all_predictions.keys()}
vars_95 = {model: {} for model in all_predictions.keys()}
turnovers = {model: {} for model in all_predictions.keys()}

# Evaluate performance for each model and factor combination and plot the results
for model_name in all_predictions.keys():
    for n_factors in tqdm(all_predictions[model_name].keys(), desc=f"Processing factors for {model_name}"):
        plt.figure(figsize=(10, 6))
        predictions_df = all_predictions[model_name][n_factors]
        performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover = evaluate_performance(predictions_df, original_df)
        sharpe_ratios[model_name][n_factors] = sharpe_ratio  # Directly using the annualized Sharpe ratio
        avg_returns[model_name][n_factors] = avg_return
        volatilities[model_name][n_factors] = volatility
        vars_95[model_name][n_factors] = var_95
        turnovers[model_name][n_factors] = turnover
        plt.plot(performance_df.index, performance_df['Cumulative_Returns'], label=f'{model_name} (Sharpe: {sharpe_ratio:.2f})')
        plt.title(f'Cumulative Returns for Factor {n_factors} - {model_name}')
        plt.xlabel('Year')
        plt.ylabel('Cumulative Returns')
        plt.legend()
        plt.show()

# Create DataFrames for each metric
sharpe_df = pd.DataFrame(sharpe_ratios).T
avg_returns_df = pd.DataFrame(avg_returns).T
volatilities_df = pd.DataFrame(volatilities).T
vars_95_df = pd.DataFrame(vars_95).T
turnovers_df = pd.DataFrame(turnovers).T

# Convert DataFrames to LaTeX tables
latex_sharpe = sharpe_df.to_latex(index=True, header=True, float_format="%.2f")
latex_avg_returns = avg_returns_df.to_latex(index=True, header=True, float_format="%.2f")
latex_volatilities = volatilities_df.to_latex(index=True, header=True, float_format="%.2f")
latex_vars_95 = vars_95_df.to_latex(index=True, header=True, float_format="%.2f")
latex_turnovers = turnovers_df.to_latex(index=True, header=True, float_format="%.2f")

# Print the LaTeX tables
print("Sharpe Ratios:")
print(latex_sharpe)
print("\nAverage Returns:")
print(latex_avg_returns)
print("\nVolatilities:")
print(latex_volatilities)
print("\nValue-At-Risk (95%):")
print(latex_vars_95)
print("\nTurnovers:")
print(latex_turnovers)


**Monthly Annualized LS sharpe ratio, value weighted LS portfolio returns TIME T**

In [ ]:
import pandas as pd
from pathlib import Path
from google.colab import drive
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

# Mount Google Drive
drive.mount('/content/drive')

# Load the original dataset
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/stock_characteristics_monthly_py.parquet'
original_df = pd.read_parquet(file_path)

# Rename the 'month' column to 'date'
original_df = original_df.rename(columns={'month': 'date'})

# Convert date columns to the same format
original_df['date'] = pd.to_datetime(original_df['date'])

# Define the base paths for different models
base_paths = [
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_0'),
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_1'),
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_2'),
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3')
]

def load_predictions(hdf5_path):
    predictions_by_n_factors = {}
    with pd.HDFStore(hdf5_path, mode='r') as store:
        keys = store.keys()
        for key in keys:
            if key.startswith('/n_factors_'):
                n_factors = key.split('_')[2]
                predictions_df = store.get(key)
                # Reset the index to make 'date' and 'permno' regular columns
                predictions_df = predictions_df.reset_index()
                predictions_df['date'] = pd.to_datetime(predictions_df['date'])
                predictions_df['permno'] = predictions_df['permno'].astype(int)
                predictions_by_n_factors[n_factors] = predictions_df
    return predictions_by_n_factors

# Load predictions for all models
all_predictions = {}
for base_path in base_paths:
    results_path = base_path / 'results' / 'asset_pricing' / 'all_predictions.h5'
    model_name = base_path.name  # Use the base directory name as the model name
    all_predictions[model_name] = load_predictions(str(results_path))

# Function to calculate value-weighted decile portfolios
def calculate_deciles(predicted, actual, mktcap_lag):
    deciles = pd.qcut(predicted.rank(method='first'), 10, labels=False) + 1
    top_decile = actual[deciles == 10]
    bottom_decile = actual[deciles == 1]
    top_weights = mktcap_lag[deciles == 10]
    bottom_weights = mktcap_lag[deciles == 1]
    top_return = np.average(top_decile, weights=top_weights)
    bottom_return = np.average(bottom_decile, weights=bottom_weights)
    zero_net_investment_return = top_return - bottom_return
    return zero_net_investment_return

# Function to calculate additional performance metrics
def calculate_metrics(returns):
    avg_return = returns.mean()
    volatility = returns.std()
    var_95 = returns.quantile(0.05)
    turnover = (returns.diff().abs().sum()) / (2 * returns.abs().sum())
    return avg_return, volatility, var_95, turnover

# Function to evaluate economic performance for each model
def evaluate_performance(predictions_df, original_df):
    predictions_df = predictions_df.sort_values(by='date')
    merged_df = pd.merge(predictions_df, original_df[['date', 'permno', 'mktcap_lag']], on=['date', 'permno'])
    merged_df['month'] = merged_df['date'].dt.to_period('M')
    monthly_returns = merged_df.groupby('month').apply(lambda x: calculate_deciles(x['y_pred'], x['y_true'], x['mktcap_lag']))

    performance_df = pd.DataFrame({
        'Zero_Net_Investment_Returns': monthly_returns
    })
    performance_df['Cumulative_Returns'] = (1 + performance_df['Zero_Net_Investment_Returns']).cumprod() - 1

    performance_df.index = performance_df.index.to_timestamp()  # Convert PeriodIndex to Timestamp

    sharpe_ratio = performance_df['Zero_Net_Investment_Returns'].mean() / performance_df['Zero_Net_Investment_Returns'].std()

    avg_return, volatility, var_95, turnover = calculate_metrics(performance_df['Zero_Net_Investment_Returns'])

    return performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover

# Dictionaries to store metrics
sharpe_ratios = {model: {} for model in all_predictions.keys()}
avg_returns = {model: {} for model in all_predictions.keys()}
volatilities = {model: {} for model in all_predictions.keys()}
vars_95 = {model: {} for model in all_predictions.keys()}
turnovers = {model: {} for model in all_predictions.keys()}

# Evaluate performance for each model and factor combination and plot the results
for model_name in all_predictions.keys():
    for n_factors in tqdm(all_predictions[model_name].keys(), desc=f"Processing factors for {model_name}"):
        plt.figure(figsize=(10, 6))
        predictions_df = all_predictions[model_name][n_factors]
        performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover = evaluate_performance(predictions_df, original_df)
        sharpe_ratio_annualized = sharpe_ratio * np.sqrt(12)  # Annualize the Sharpe ratio
        sharpe_ratios[model_name][n_factors] = sharpe_ratio_annualized
        avg_returns[model_name][n_factors] = avg_return
        volatilities[model_name][n_factors] = volatility
        vars_95[model_name][n_factors] = var_95
        turnovers[model_name][n_factors] = turnover
        plt.plot(performance_df.index, performance_df['Cumulative_Returns'], label=f'{model_name} (Sharpe: {sharpe_ratio_annualized:.2f})')
        plt.title(f'Cumulative Returns for Factor {n_factors} - {model_name}')
        plt.xlabel('Month')
        plt.ylabel('Cumulative Returns')
        plt.legend()
        plt.show()

# Create DataFrames for each metric
sharpe_df = pd.DataFrame(sharpe_ratios).T
avg_returns_df = pd.DataFrame(avg_returns).T
volatilities_df = pd.DataFrame(volatilities).T
vars_95_df = pd.DataFrame(vars_95).T
turnovers_df = pd.DataFrame(turnovers).T

# Convert DataFrames to LaTeX tables
latex_sharpe = sharpe_df.to_latex(index=True, header=True, float_format="%.2f")
latex_avg_returns = avg_returns_df.to_latex(index=True, header=True, float_format="%.2f")
latex_volatilities = volatilities_df.to_latex(index=True, header=True, float_format="%.2f")
latex_vars_95 = vars_95_df.to_latex(index=True, header=True, float_format="%.2f")
latex_turnovers = turnovers_df.to_latex(index=True, header=True, float_format="%.2f")

# Print the LaTeX tables
print("Sharpe Ratios:")
print(latex_sharpe)
print("\nAverage Returns:")
print(latex_avg_returns)
print("\nVolatilities:")
print(latex_volatilities)
print("\nValue-At-Risk (95%):")
print(latex_vars_95)
print("\nTurnovers:")
print(latex_turnovers)


In [ ]:
import pandas as pd
from pathlib import Path
from google.colab import drive
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

# Mount Google Drive
drive.mount('/content/drive')

# Load the original dataset
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/stock_characteristics_monthly_py.parquet'
original_df = pd.read_parquet(file_path)

# Rename the 'month' column to 'date'
original_df = original_df.rename(columns={'month': 'date'})

# Convert date columns to the same format
original_df['date'] = pd.to_datetime(original_df['date'])

# Define the base paths for different models
base_paths = [
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_0_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_1_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_2_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3_ret_excess_fwd'
]

def load_predictions(hdf5_path):
    predictions_by_n_factors = {}
    with pd.HDFStore(hdf5_path, mode='r') as store:
        keys = store.keys()
        for key in keys:
            if key.startswith('/n_factors_'):
                n_factors = key.split('_')[2]
                predictions_df = store.get(key)
                # Reset the index to make 'date' and 'permno' regular columns
                predictions_df = predictions_df.reset_index()
                predictions_df['date'] = pd.to_datetime(predictions_df['date'])
                predictions_df['permno'] = predictions_df['permno'].astype(int)
                predictions_by_n_factors[n_factors] = predictions_df
    return predictions_by_n_factors

# Load predictions for all models
all_predictions = {}
for base_path_str in base_paths:
    base_path = Path(base_path_str)
    results_path = base_path / 'results' / 'asset_pricing' / 'all_predictions.h5'
    model_name = base_path.parts[-1]  # Use the full directory name as the model name
    all_predictions[model_name] = load_predictions(str(results_path))

# Function to calculate value-weighted decile portfolios
def calculate_deciles(predicted, actual, mktcap_lag):
    deciles = pd.qcut(predicted.rank(method='first'), 10, labels=False) + 1
    top_decile = actual[deciles == 10]
    bottom_decile = actual[deciles == 1]
    top_weights = mktcap_lag[deciles == 10]
    bottom_weights = mktcap_lag[deciles == 1]
    top_return = np.average(top_decile, weights=top_weights)
    bottom_return = np.average(bottom_decile, weights=bottom_weights)
    zero_net_investment_return = top_return - bottom_return
    return zero_net_investment_return

# Function to calculate additional performance metrics
def calculate_metrics(returns):
    avg_return = returns.mean()
    volatility = returns.std()
    var_95 = returns.quantile(0.05)
    turnover = (returns.diff().abs().sum()) / (2 * returns.abs().sum())
    return avg_return, volatility, var_95, turnover

# Function to evaluate economic performance for each model
def evaluate_performance(predictions_df, original_df):
    predictions_df = predictions_df.sort_values(by='date')
    merged_df = pd.merge(predictions_df, original_df[['date', 'permno', 'mktcap_lag']], on=['date', 'permno'])
    merged_df['month'] = merged_df['date'].dt.to_period('M')
    monthly_returns = merged_df.groupby('month').apply(lambda x: calculate_deciles(x['y_pred'], x['y_true'], x['mktcap_lag']))

    performance_df = pd.DataFrame({
        'Zero_Net_Investment_Returns': monthly_returns
    })
    performance_df['Cumulative_Returns'] = (1 + performance_df['Zero_Net_Investment_Returns']).cumprod() - 1

    performance_df.index = performance_df.index.to_timestamp()  # Convert PeriodIndex to Timestamp

    sharpe_ratio = performance_df['Zero_Net_Investment_Returns'].mean() / performance_df['Zero_Net_Investment_Returns'].std()

    avg_return, volatility, var_95, turnover = calculate_metrics(performance_df['Zero_Net_Investment_Returns'])

    return performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover

# Dictionaries to store metrics
sharpe_ratios = {model: {} for model in all_predictions.keys()}
avg_returns = {model: {} for model in all_predictions.keys()}
volatilities = {model: {} for model in all_predictions.keys()}
vars_95 = {model: {} for model in all_predictions.keys()}
turnovers = {model: {} for model in all_predictions.keys()}

# Evaluate performance for each model and factor combination and plot the results
for model_name in all_predictions.keys():
    for n_factors in tqdm(all_predictions[model_name].keys(), desc=f"Processing factors for {model_name}"):
        plt.figure(figsize=(10, 6))
        predictions_df = all_predictions[model_name][n_factors]
        performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover = evaluate_performance(predictions_df, original_df)
        sharpe_ratios[model_name][n_factors] = sharpe_ratio
        avg_returns[model_name][n_factors] = avg_return
        volatilities[model_name][n_factors] = volatility
        vars_95[model_name][n_factors] = var_95
        turnovers[model_name][n_factors] = turnover
        plt.plot(performance_df.index, performance_df['Cumulative_Returns'], label=f'{model_name} (Sharpe: {sharpe_ratio:.2f})')
        plt.title(f'Cumulative Returns for Factor {n_factors} - {model_name}')
        plt.xlabel('Month')
        plt.ylabel('Cumulative Returns')
        plt.legend()
        plt.show()

# Create DataFrames for each metric
sharpe_df = pd.DataFrame(sharpe_ratios).T
avg_returns_df = pd.DataFrame(avg_returns).T
volatilities_df = pd.DataFrame(volatilities).T
vars_95_df = pd.DataFrame(vars_95).T
turnovers_df = pd.DataFrame(turnovers).T

# Convert DataFrames to LaTeX tables
latex_sharpe = sharpe_df.to_latex(index=True, header=True, float_format="%.2f")
latex_avg_returns = avg_returns_df.to_latex(index=True, header=True, float_format="%.2f")
latex_volatilities = volatilities_df.to_latex(index=True, header=True, float_format="%.2f")
latex_vars_95 = vars_95_df.to_latex(index=True, header=True, float_format="%.2f")
latex_turnovers = turnovers_df.to_latex(index=True, header=True, float_format="%.2f")

# Print the LaTeX tables
print("Sharpe Ratios:")
print(latex_sharpe)
print("\nAverage Returns:")
print(latex_avg_returns)
print("\nVolatilities:")
print(latex_volatilities)
print("\nValue-At-Risk (95%):")
print(latex_vars_95)
print("\nTurnovers:")
print(latex_turnovers)


**Value weighted port LS portfolios MONTHLY annualized sharpe ratio (forward returns)**

In [ ]:
import pandas as pd
from pathlib import Path
from google.colab import drive
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

# Mount Google Drive
drive.mount('/content/drive')

# Load the original dataset
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/stock_characteristics_monthly_py.parquet'
original_df = pd.read_parquet(file_path)

# Rename the 'month' column to 'date'
original_df = original_df.rename(columns={'month': 'date'})

# Convert date columns to the same format
original_df['date'] = pd.to_datetime(original_df['date'])

# Define the base paths for different models
base_paths = [
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_0_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_1_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_2_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3_ret_excess_fwd'
]

def load_predictions(hdf5_path):
    predictions_by_n_factors = {}
    with pd.HDFStore(hdf5_path, mode='r') as store:
        keys = store.keys()
        for key in keys:
            if key.startswith('/n_factors_'):
                n_factors = key.split('_')[2]
                predictions_df = store.get(key)
                # Reset the index to make 'date' and 'permno' regular columns
                predictions_df = predictions_df.reset_index()
                predictions_df['date'] = pd.to_datetime(predictions_df['date'])
                predictions_df['permno'] = predictions_df['permno'].astype(int)
                predictions_by_n_factors[n_factors] = predictions_df
    return predictions_by_n_factors

# Load predictions for all models
all_predictions = {}
for base_path_str in base_paths:
    base_path = Path(base_path_str)
    results_path = base_path / 'results' / 'asset_pricing' / 'all_predictions.h5'
    model_name = base_path.parts[-1]  # Use the full directory name as the model name
    all_predictions[model_name] = load_predictions(str(results_path))

# Function to calculate value-weighted decile portfolios
def calculate_deciles(predicted, actual, mktcap_lag):
    deciles = pd.qcut(predicted.rank(method='first'), 10, labels=False) + 1
    top_decile = actual[deciles == 10]
    bottom_decile = actual[deciles == 1]
    top_weights = mktcap_lag[deciles == 10]
    bottom_weights = mktcap_lag[deciles == 1]
    top_return = np.average(top_decile, weights=top_weights)
    bottom_return = np.average(bottom_decile, weights=bottom_weights)
    zero_net_investment_return = top_return - bottom_return
    return zero_net_investment_return

# Function to calculate additional performance metrics
def calculate_metrics(returns):
    avg_return = returns.mean()
    volatility = returns.std()
    var_95 = returns.quantile(0.05)
    turnover = (returns.diff().abs().sum()) / (2 * returns.abs().sum())
    return avg_return, volatility, var_95, turnover

# Function to evaluate economic performance for each model
def evaluate_performance(predictions_df, original_df):
    predictions_df = predictions_df.sort_values(by='date')
    merged_df = pd.merge(predictions_df, original_df[['date', 'permno', 'mktcap_lag']], on=['date', 'permno'])
    merged_df['month'] = merged_df['date'].dt.to_period('M')
    monthly_returns = merged_df.groupby('month').apply(lambda x: calculate_deciles(x['y_pred'], x['y_true'], x['mktcap_lag']))

    performance_df = pd.DataFrame({
        'Zero_Net_Investment_Returns': monthly_returns
    })
    performance_df['Cumulative_Returns'] = (1 + performance_df['Zero_Net_Investment_Returns']).cumprod() - 1

    performance_df.index = performance_df.index.to_timestamp()  # Convert PeriodIndex to Timestamp

    sharpe_ratio = performance_df['Zero_Net_Investment_Returns'].mean() / performance_df['Zero_Net_Investment_Returns'].std()

    avg_return, volatility, var_95, turnover = calculate_metrics(performance_df['Zero_Net_Investment_Returns'])

    return performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover

# Dictionaries to store metrics
sharpe_ratios = {model: {} for model in all_predictions.keys()}
avg_returns = {model: {} for model in all_predictions.keys()}
volatilities = {model: {} for model in all_predictions.keys()}
vars_95 = {model: {} for model in all_predictions.keys()}
turnovers = {model: {} for model in all_predictions.keys()}

# Evaluate performance for each model and factor combination and plot the results
for model_name in all_predictions.keys():
    for n_factors in tqdm(all_predictions[model_name].keys(), desc=f"Processing factors for {model_name}"):
        plt.figure(figsize=(10, 6))
        predictions_df = all_predictions[model_name][n_factors]
        performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover = evaluate_performance(predictions_df, original_df)
        sharpe_ratio_annualized = sharpe_ratio * np.sqrt(12)  # Annualize the Sharpe ratio
        sharpe_ratios[model_name][n_factors] = sharpe_ratio_annualized
        avg_returns[model_name][n_factors] = avg_return
        volatilities[model_name][n_factors] = volatility
        vars_95[model_name][n_factors] = var_95
        turnovers[model_name][n_factors] = turnover
        plt.plot(performance_df.index, performance_df['Cumulative_Returns'], label=f'{model_name} (Sharpe: {sharpe_ratio_annualized:.2f})')
        plt.title(f'Cumulative Returns for Factor {n_factors} - {model_name}')
        plt.xlabel('Month')
        plt.ylabel('Cumulative Returns')
        plt.legend()
        plt.show()

# Create DataFrames for each metric
sharpe_df = pd.DataFrame(sharpe_ratios).T
avg_returns_df = pd.DataFrame(avg_returns).T
volatilities_df = pd.DataFrame(volatilities).T
vars_95_df = pd.DataFrame(vars_95).T
turnovers_df = pd.DataFrame(turnovers).T

# Convert DataFrames to LaTeX tables
latex_sharpe = sharpe_df.to_latex(index=True, header=True, float_format="%.2f")
latex_avg_returns = avg_returns_df.to_latex(index=True, header=True, float_format="%.2f")
latex_volatilities = volatilities_df.to_latex(index=True, header=True, float_format="%.2f")
latex_vars_95 = vars_95_df.to_latex(index=True, header=True, float_format="%.2f")
latex_turnovers = turnovers_df.to_latex(index=True, header=True, float_format="%.2f")

# Print the LaTeX tables
print("Sharpe Ratios:")
print(latex_sharpe)
print("\nAverage Returns:")
print(latex_avg_returns)
print("\nVolatilities:")
print(latex_volatilities)
print("\nValue-At-Risk (95%):")
print(latex_vars_95)
print("\nTurnovers:")
print(latex_turnovers)


**annualized SR for Equal-weighted LS portfolio for MONTHLY returns (forward returns)**

In [ ]:
import pandas as pd
from pathlib import Path
from google.colab import drive
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

# Mount Google Drive
drive.mount('/content/drive')

# Load the original dataset
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/stock_characteristics_monthly_py.parquet'
original_df = pd.read_parquet(file_path)

# Rename the 'month' column to 'date'
original_df = original_df.rename(columns={'month': 'date'})

# Convert date columns to the same format
original_df['date'] = pd.to_datetime(original_df['date'])

# Define the base paths for different models
base_paths = [
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_0_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_1_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_2_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3_ret_excess_fwd'
]

def load_predictions(hdf5_path):
    predictions_by_n_factors = {}
    with pd.HDFStore(hdf5_path, mode='r') as store:
        keys = store.keys()
        for key in keys:
            if key.startswith('/n_factors_'):
                n_factors = key.split('_')[2]
                predictions_df = store.get(key)
                # Reset the index to make 'date' and 'permno' regular columns
                predictions_df = predictions_df.reset_index()
                predictions_df['date'] = pd.to_datetime(predictions_df['date'])
                predictions_df['permno'] = predictions_df['permno'].astype(int)
                predictions_by_n_factors[n_factors] = predictions_df
    return predictions_by_n_factors

# Load predictions for all models
all_predictions = {}
for base_path_str in base_paths:
    base_path = Path(base_path_str)
    results_path = base_path / 'results' / 'asset_pricing' / 'all_predictions.h5'
    model_name = base_path.parts[-1]  # Use the full directory name as the model name
    all_predictions[model_name] = load_predictions(str(results_path))

# Function to calculate equal-weighted decile portfolios
def calculate_deciles(predicted, actual):
    deciles = pd.qcut(predicted.rank(method='first'), 10, labels=False) + 1
    top_decile = actual[deciles == 10]
    bottom_decile = actual[deciles == 1]
    top_return = top_decile.mean()
    bottom_return = bottom_decile.mean()
    zero_net_investment_return = top_return - bottom_return
    return zero_net_investment_return

# Function to calculate additional performance metrics
def calculate_metrics(returns):
    avg_return = returns.mean()
    volatility = returns.std()
    var_95 = returns.quantile(0.05)
    turnover = (returns.diff().abs().sum()) / (2 * returns.abs().sum())
    return avg_return, volatility, var_95, turnover

# Function to evaluate economic performance for each model
def evaluate_performance(predictions_df, original_df):
    predictions_df = predictions_df.sort_values(by='date')
    merged_df = pd.merge(predictions_df, original_df[['date', 'permno', 'mktcap_lag']], on=['date', 'permno'])
    merged_df['month'] = merged_df['date'].dt.to_period('M')
    monthly_returns = merged_df.groupby('month').apply(lambda x: calculate_deciles(x['y_pred'], x['y_true']))

    performance_df = pd.DataFrame({
        'Zero_Net_Investment_Returns': monthly_returns
    })
    performance_df['Cumulative_Returns'] = (1 + performance_df['Zero_Net_Investment_Returns']).cumprod() - 1

    performance_df.index = performance_df.index.to_timestamp()  # Convert PeriodIndex to Timestamp

    sharpe_ratio = performance_df['Zero_Net_Investment_Returns'].mean() / performance_df['Zero_Net_Investment_Returns'].std()

    avg_return, volatility, var_95, turnover = calculate_metrics(performance_df['Zero_Net_Investment_Returns'])

    return performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover

# Dictionaries to store metrics
sharpe_ratios = {model: {} for model in all_predictions.keys()}
avg_returns = {model: {} for model in all_predictions.keys()}
volatilities = {model: {} for model in all_predictions.keys()}
vars_95 = {model: {} for model in all_predictions.keys()}
turnovers = {model: {} for model in all_predictions.keys()}

# Evaluate performance for each model and factor combination and plot the results
for model_name in all_predictions.keys():
    for n_factors in tqdm(all_predictions[model_name].keys(), desc=f"Processing factors for {model_name}"):
        plt.figure(figsize=(10, 6))
        predictions_df = all_predictions[model_name][n_factors]
        performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover = evaluate_performance(predictions_df, original_df)
        sharpe_ratio_annualized = sharpe_ratio * np.sqrt(12)  # Annualize the Sharpe ratio
        sharpe_ratios[model_name][n_factors] = sharpe_ratio_annualized
        avg_returns[model_name][n_factors] = avg_return
        volatilities[model_name][n_factors] = volatility
        vars_95[model_name][n_factors] = var_95
        turnovers[model_name][n_factors] = turnover
        plt.plot(performance_df.index, performance_df['Cumulative_Returns'], label=f'{model_name} (Sharpe: {sharpe_ratio_annualized:.2f})')
        plt.title(f'Cumulative Returns for Factor {n_factors} - {model_name}')
        plt.xlabel('Month')
        plt.ylabel('Cumulative Returns')
        plt.legend()
        plt.show()

# Create DataFrames for each metric
sharpe_df = pd.DataFrame(sharpe_ratios).T
avg_returns_df = pd.DataFrame(avg_returns).T
volatilities_df = pd.DataFrame(volatilities).T
vars_95_df = pd.DataFrame(vars_95).T
turnovers_df = pd.DataFrame(turnovers).T

# Convert DataFrames to LaTeX tables
latex_sharpe = sharpe_df.to_latex(index=True, header=True, float_format="%.2f")
latex_avg_returns = avg_returns_df.to_latex(index=True, header=True, float_format="%.2f")
latex_volatilities = volatilities_df.to_latex(index=True, header=True, float_format="%.2f")
latex_vars_95 = vars_95_df.to_latex(index=True, header=True, float_format="%.2f")
latex_turnovers = turnovers_df.to_latex(index=True, header=True, float_format="%.2f")

# Print the LaTeX tables
print("Sharpe Ratios:")
print(latex_sharpe)
print("\nAverage Returns:")
print(latex_avg_returns)
print("\nVolatilities:")
print(latex_volatilities)
print("\nValue-At-Risk (95%):")
print(latex_vars_95)
print("\nTurnovers:")
print(latex_turnovers)


In [ ]:
print(all_predictions)


In [ ]:
import pandas as pd
from pathlib import Path
from google.colab import drive
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

# Mount Google Drive
drive.mount('/content/drive')

# Load the original dataset
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/stock_characteristics_monthly_py.parquet'
original_df = pd.read_parquet(file_path)

# Rename the 'month' column to 'date'
original_df = original_df.rename(columns={'month': 'date'})
original_df['date'] = pd.to_datetime(original_df['date'])

# Define the base paths for different models
base_paths = [
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_0'),
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_1'),
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_2'),
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3')
]

def load_predictions(hdf5_path):
    predictions_by_n_factors = {}
    with pd.HDFStore(hdf5_path, mode='r') as store:
        keys = store.keys()
        for key in keys:
            if key.startswith('/n_factors_'):
                n_factors = key.split('_')[2]
                predictions_df = store.get(key)
                predictions_df = predictions_df.reset_index()
                predictions_df['date'] = pd.to_datetime(predictions_df['date'])
                predictions_df['permno'] = predictions_df['permno'].astype(int)
                predictions_by_n_factors[n_factors] = predictions_df
    return predictions_by_n_factors

# Load predictions for all models
all_predictions = {}
for base_path in base_paths:
    results_path = base_path / 'results' / 'asset_pricing' / 'all_predictions.h5'
    model_name = base_path.parts[-1].split('_')[-1]
    all_predictions[model_name] = load_predictions(str(results_path))

# Function to calculate value-weighted decile portfolios
def calculate_deciles(predicted, actual, mktcap_lag):
    deciles = pd.qcut(predicted.rank(method='first'), 10, labels=False) + 1
    top_decile = actual[deciles == 10]
    bottom_decile = actual[deciles == 1]
    top_weights = mktcap_lag[deciles == 10]
    bottom_weights = mktcap_lag[deciles == 1]
    top_return = np.average(top_decile, weights=top_weights)
    bottom_return = np.average(bottom_decile, weights=bottom_weights)
    zero_net_investment_return = top_return - bottom_return
    return zero_net_investment_return

# Function to calculate additional performance metrics
def calculate_metrics(returns):
    avg_return = returns.mean()
    volatility = returns.std()
    var_95 = returns.quantile(0.05)
    turnover = (returns.diff().abs().sum()) / (2 * returns.abs().sum())
    return avg_return, volatility, var_95, turnover

# Function to evaluate economic performance for each model
def evaluate_performance(predictions_df, original_df):
    predictions_df = predictions_df.sort_values(by='date')
    merged_df = pd.merge(predictions_df, original_df[['date', 'permno', 'mktcap_lag']], on=['date', 'permno'])
    merged_df['year'] = merged_df['date'].dt.year
    annual_returns = merged_df.groupby('year').apply(lambda x: calculate_deciles(x['y_pred'], x['y_true'], x['mktcap_lag']))

    performance_df = pd.DataFrame({
        'Zero_Net_Investment_Returns': annual_returns
    })
    performance_df['Cumulative_Returns'] = (1 + performance_df['Zero_Net_Investment_Returns']).cumprod() - 1

    sharpe_ratio = performance_df['Zero_Net_Investment_Returns'].mean() / performance_df['Zero_Net_Investment_Returns'].std()

    avg_return, volatility, var_95, turnover = calculate_metrics(performance_df['Zero_Net_Investment_Returns'])

    return performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover

# Example S&P 500 permnos (This should be your actual list of S&P 500 permnos)
s_and_p_permnos = original_df['permno'].unique()  # Assuming all permnos are part of S&P 500 for this example

# Function to calculate S&P 500 returns
def calculate_sp500_returns(predictions_df, original_df, s_and_p_permnos):
    # Filter the dataframe for S&P 500 stocks
    sp500_df = original_df[original_df['permno'].isin(s_and_p_permnos)]

    # Merge predictions with S&P 500 stock data
    merged_df = pd.merge(predictions_df, sp500_df[['date', 'permno', 'mktcap_lag']], on=['date', 'permno'], how='inner')

    # Calculate value-weighted returns
    def weighted_return(df):
        weights = df['mktcap_lag'] / df['mktcap_lag'].sum()
        return np.sum(weights * df['y_pred'])

    # Group by date to calculate the S&P 500 return for each month
    sp500_returns = merged_df.groupby('date').apply(weighted_return)

    return sp500_returns

# Process one set of predictions (example using model '0' and factor '1')
model = '0'
n_factors = '1.0'
predictions_df = all_predictions[model][n_factors]

# Evaluate performance for the model
performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover = evaluate_performance(predictions_df, original_df)

# Convert performance_df yearly index to datetime index
performance_df.index = pd.to_datetime(performance_df.index, format='%Y') + pd.offsets.YearEnd()

# Calculate S&P 500 returns
sp500_returns = calculate_sp500_returns(predictions_df, original_df, s_and_p_permnos)
sp500_cumulative_returns = (1 + sp500_returns).cumprod() - 1

# Resample S&P 500 returns to yearly
sp500_cumulative_returns_yearly = sp500_cumulative_returns.resample('Y').last()

# Plot the results
plt.figure(figsize=(10, 6))
plt.plot(performance_df.index, performance_df['Cumulative_Returns'], label=f'Model {model} (Sharpe: {sharpe_ratio:.2f})')
plt.plot(sp500_cumulative_returns_yearly.index, sp500_cumulative_returns_yearly, label='S&P 500')
plt.title(f'Cumulative Returns for Model {model} and Factor {n_factors}')
plt.xlabel('Year')
plt.ylabel('Cumulative Returns')
plt.legend()
plt.show()

# Print metrics
print(f"Model {model}, Factor {n_factors}")
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
print(f"Average Return: {avg_return:.2%}")
print(f"Volatility: {volatility:.2%}")
print(f"Value-at-Risk (95%): {var_95:.2%}")
print(f"Turnover: {turnover:.2%}")

In [ ]:
import pandas as pd
from pathlib import Path
from google.colab import drive
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

# Mount Google Drive
drive.mount('/content/drive')

# Load the original dataset
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/stock_characteristics_monthly_py.parquet'
original_df = pd.read_parquet(file_path)

# Rename the 'month' column to 'date'
original_df = original_df.rename(columns={'month': 'date'})

# Convert date columns to the same format
original_df['date'] = pd.to_datetime(original_df['date'])

# Define the base paths for different models
base_paths = [
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_2_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_1_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_0_ret_excess_fwd'
]

# Function to load predictions from a specific HDF5 file
def load_predictions(hdf5_path):
    predictions_by_n_factors = {}
    with pd.HDFStore(hdf5_path, mode='r') as store:
        keys = store.keys()
        for key in keys:
            if key.startswith('/n_factors_'):
                n_factors = key.split('_')[2]
                predictions_df = store.get(key)
                predictions_df = predictions_df.reset_index()
                predictions_df['date'] = pd.to_datetime(predictions_df['date'])
                predictions_df['permno'] = predictions_df['permno'].astype(int)
                predictions_by_n_factors[n_factors] = predictions_df
    return predictions_by_n_factors

# Load predictions for all models
all_predictions = {}
for base_path_str in base_paths:
    base_path = Path(base_path_str)
    results_path = base_path / 'results' / 'asset_pricing' / 'all_predictions.h5'
    model_name = base_path.parts[-1].split('_')[-1]
    all_predictions[model_name] = load_predictions(str(results_path))

# Function to calculate value-weighted decile portfolios
def calculate_deciles(predicted, actual, mktcap_lag):
    deciles = pd.qcut(predicted.rank(method='first'), 10, labels=False) + 1
    top_decile = actual[deciles == 10]
    bottom_decile = actual[deciles == 1]
    top_weights = mktcap_lag[deciles == 10]
    bottom_weights = mktcap_lag[deciles == 1]
    top_return = np.average(top_decile, weights=top_weights)
    bottom_return = np.average(bottom_decile, weights=bottom_weights)
    zero_net_investment_return = top_return - bottom_return
    return zero_net_investment_return

# Function to calculate additional performance metrics
def calculate_metrics(returns):
    avg_return = returns.mean()
    volatility = returns.std()
    var_95 = returns.quantile(0.05)
    turnover = (returns.diff().abs().sum()) / (2 * returns.abs().sum())
    return avg_return, volatility, var_95, turnover

# Function to evaluate economic performance for each model
def evaluate_performance(predictions_df, original_df):
    predictions_df = predictions_df.sort_values(by='date')
    merged_df = pd.merge(predictions_df, original_df[['date', 'permno', 'mktcap_lag']], on=['date', 'permno'])
    merged_df['year'] = merged_df['date'].dt.year
    annual_returns = merged_df.groupby('year').apply(lambda x: calculate_deciles(x['y_pred'], x['y_true'], x['mktcap_lag']))

    performance_df = pd.DataFrame({
        'Zero_Net_Investment_Returns': annual_returns
    })
    performance_df['Cumulative_Returns'] = (1 + performance_df['Zero_Net_Investment_Returns']).cumprod() - 1

    sharpe_ratio = performance_df['Zero_Net_Investment_Returns'].mean() / performance_df['Zero_Net_Investment_Returns'].std()

    avg_return, volatility, var_95, turnover = calculate_metrics(performance_df['Zero_Net_Investment_Returns'])

    return performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover

# Dictionaries to store metrics
sharpe_ratios = {model: {} for model in all_predictions.keys()}
avg_returns = {model: {} for model in all_predictions.keys()}
volatilities = {model: {} for model in all_predictions.keys()}
vars_95 = {model: {} for model in all_predictions.keys()}
turnovers = {model: {} for model in all_predictions.keys()}

# Evaluate performance for each model and factor combination and plot the results
for n_factors in tqdm(all_predictions['fwd'].keys(), desc="Processing factors"):
    plt.figure(figsize=(10, 6))
    for model in all_predictions.keys():
        predictions_df = all_predictions[model][n_factors]
        performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover = evaluate_performance(predictions_df, original_df)
        sharpe_ratios[model][n_factors] = sharpe_ratio
        avg_returns[model][n_factors] = avg_return
        volatilities[model][n_factors] = volatility
        vars_95[model][n_factors] = var_95
        turnovers[model][n_factors] = turnover
        plt.plot(performance_df.index, performance_df['Cumulative_Returns'], label=f'{model} (Sharpe: {sharpe_ratio:.2f})')
    plt.title(f'Cumulative Returns for Factor {n_factors}')
    plt.xlabel('Year')
    plt.ylabel('Cumulative Returns')
    plt.legend()
    plt.show()

# Create DataFrames for each metric
sharpe_df = pd.DataFrame(sharpe_ratios).T
avg_returns_df = pd.DataFrame(avg_returns).T
volatilities_df = pd.DataFrame(volatilities).T
vars_95_df = pd.DataFrame(vars_95).T
turnovers_df = pd.DataFrame(turnovers).T

# Convert DataFrames to LaTeX tables
latex_sharpe = sharpe_df.to_latex(index=True, header=True, float_format="%.2f")
latex_avg_returns = avg_returns_df.to_latex(index=True, header=True, float_format="%.2f")
latex_volatilities = volatilities_df.to_latex(index=True, header=True, float_format="%.2f")
latex_vars_95 = vars_95_df.to_latex(index=True, header=True, float_format="%.2f")
latex_turnovers = turnovers_df.to_latex(index=True, header=True, float_format="%.2f")

# Print the LaTeX tables
print("Sharpe Ratios:")
print(latex_sharpe)
print("\nAverage Returns:")
print(latex_avg_returns)
print("\nVolatilities:")
print(latex_volatilities)
print("\nValue-At-Risk (95%):")
print(latex_vars_95)
print("\nTurnovers:")
print(latex_turnovers)


In [ ]:
import pandas as pd
from pathlib import Path
from google.colab import drive
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

# Mount Google Drive
drive.mount('/content/drive')

# Load the original dataset
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/stock_characteristics_monthly_py.parquet'
original_df = pd.read_parquet(file_path)

# Rename the 'month' column to 'date'
original_df = original_df.rename(columns={'month': 'date'})
original_df['date'] = pd.to_datetime(original_df['date'])

# Define the base paths for different models
base_paths = [
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_0'),
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_1'),
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_2'),
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3')
]

def load_predictions(hdf5_path):
    predictions_by_n_factors = {}
    with pd.HDFStore(hdf5_path, mode='r') as store:
        keys = store.keys()
        for key in keys:
            if key.startswith('/n_factors_'):
                n_factors = key.split('_')[2]
                predictions_df = store.get(key)
                predictions_df = predictions_df.reset_index()
                predictions_df['date'] = pd.to_datetime(predictions_df['date'])
                predictions_df['permno'] = predictions_df['permno'].astype(int)
                predictions_by_n_factors[n_factors] = predictions_df
    return predictions_by_n_factors

# Load predictions for all models
all_predictions = {}
for base_path in base_paths:
    results_path = base_path / 'results' / 'asset_pricing' / 'all_predictions.h5'
    model_name = base_path.parts[-1].split('_')[-1]
    all_predictions[model_name] = load_predictions(str(results_path))

# Function to calculate value-weighted decile portfolios
def calculate_deciles(predicted, actual, mktcap_lag):
    deciles = pd.qcut(predicted.rank(method='first'), 10, labels=False) + 1
    top_decile = actual[deciles == 10]
    bottom_decile = actual[deciles == 1]
    top_weights = mktcap_lag[deciles == 10]
    bottom_weights = mktcap_lag[deciles == 1]
    top_return = np.average(top_decile, weights=top_weights)
    bottom_return = np.average(bottom_decile, weights=bottom_weights)
    zero_net_investment_return = top_return - bottom_return
    return top_return, bottom_return, zero_net_investment_return

# Function to calculate additional performance metrics
def calculate_metrics(returns):
    avg_return = returns.mean()
    volatility = returns.std()
    var_95 = returns.quantile(0.05)
    turnover = (returns.diff().abs().sum()) / (2 * returns.abs().sum())
    return avg_return, volatility, var_95, turnover

# Function to evaluate economic performance for each model
def evaluate_performance(predictions_df, original_df):
    predictions_df = predictions_df.sort_values(by='date')
    merged_df = pd.merge(predictions_df, original_df[['date', 'permno', 'mktcap_lag']], on=['date', 'permno'])
    merged_df['year'] = merged_df['date'].dt.year
    annual_returns = merged_df.groupby('year').apply(lambda x: calculate_deciles(x['y_pred'], x['y_true'], x['mktcap_lag']))

    top_returns = annual_returns.apply(lambda x: x[0])
    bottom_returns = annual_returns.apply(lambda x: x[1])
    zero_net_investment_returns = annual_returns.apply(lambda x: x[2])

    performance_df = pd.DataFrame({
        'Top_Decile_Returns': top_returns,
        'Bottom_Decile_Returns': bottom_returns,
        'Zero_Net_Investment_Returns': zero_net_investment_returns
    })
    performance_df['Cumulative_Top_Returns'] = (1 + performance_df['Top_Decile_Returns']).cumprod() - 1
    performance_df['Cumulative_Bottom_Returns'] = (1 + performance_df['Bottom_Decile_Returns']).cumprod() - 1
    performance_df['Cumulative_Zero_Net_Returns'] = (1 + performance_df['Zero_Net_Investment_Returns']).cumprod() - 1

    sharpe_ratio = performance_df['Zero_Net_Investment_Returns'].mean() / performance_df['Zero_Net_Investment_Returns'].std()

    avg_return, volatility, var_95, turnover = calculate_metrics(performance_df['Zero_Net_Investment_Returns'])

    return performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover

# Example S&P 500 permnos (This should be your actual list of S&P 500 permnos)
s_and_p_permnos = original_df['permno'].unique()  # Assuming all permnos are part of S&P 500 for this example

# Function to calculate S&P 500 returns
def calculate_sp500_returns(predictions_df, original_df, s_and_p_permnos):
    # Filter the dataframe for S&P 500 stocks
    sp500_df = original_df[original_df['permno'].isin(s_and_p_permnos)]

    # Merge predictions with S&P 500 stock data
    merged_df = pd.merge(predictions_df, sp500_df[['date', 'permno', 'mktcap_lag']], on=['date', 'permno'], how='inner')

    # Calculate value-weighted returns
    def weighted_return(df):
        weights = df['mktcap_lag'] / df['mktcap_lag'].sum()
        return np.sum(weights * df['y_pred'])

    # Group by date to calculate the S&P 500 return for each month
    sp500_returns = merged_df.groupby('date').apply(weighted_return)

    return sp500_returns

# Process one set of predictions (example using model '0' and factor '1')
model = '0'
n_factors = '1.0'
predictions_df = all_predictions[model][n_factors]

# Evaluate performance for the model
performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover = evaluate_performance(predictions_df, original_df)

# Convert performance_df yearly index to datetime index
performance_df.index = pd.to_datetime(performance_df.index, format='%Y') + pd.offsets.YearEnd()

# Calculate S&P 500 returns
sp500_returns = calculate_sp500_returns(predictions_df, original_df, s_and_p_permnos)
sp500_cumulative_returns = (1 + sp500_returns).cumprod() - 1

# Resample S&P 500 returns to yearly
sp500_cumulative_returns_yearly = sp500_cumulative_returns.resample('Y').last()

# Plot the results
plt.figure(figsize=(10, 6))
plt.plot(performance_df.index, performance_df['Cumulative_Top_Returns'], label='Top Decile Returns')
plt.plot(performance_df.index, performance_df['Cumulative_Bottom_Returns'], label='Bottom Decile Returns')
#plt.plot(performance_df.index, performance_df['Cumulative_Zero_Net_Returns'], label=f'Model {model} (Sharpe: {sharpe_ratio:.2f})')
plt.plot(sp500_cumulative_returns_yearly.index, sp500_cumulative_returns_yearly, label='S&P 500')
plt.title(f'Cumulative Returns for Model {model} and Factor {n_factors}')
plt.xlabel('Year')
plt.ylabel('Cumulative Returns')
plt.legend()
plt.show()

# Print metrics
print(f"Model {model}, Factor {n_factors}")
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
print(f"Average Return: {avg_return:.2%}")
print(f"Volatility: {volatility:.2%}")
print(f"Value-at-Risk (95%): {var_95:.2%}")
print(f"Turnover: {turnover:.2%}")


In [ ]:
import pandas as pd
from pathlib import Path
from google.colab import drive
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

# Mount Google Drive
drive.mount('/content/drive')

# Load the original dataset
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/stock_characteristics_monthly_py.parquet'
original_df = pd.read_parquet(file_path)

# Rename the 'month' column to 'date'
original_df = original_df.rename(columns={'month': 'date'})
original_df['date'] = pd.to_datetime(original_df['date'])

# Define the base paths for different models
base_paths = [
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_0'),
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_1'),
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_2'),
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3')
]

def load_predictions(hdf5_path):
    predictions_by_n_factors = {}
    with pd.HDFStore(hdf5_path, mode='r') as store:
        keys = store.keys()
        for key in keys:
            if key.startswith('/n_factors_'):
                n_factors = key.split('_')[2]
                predictions_df = store.get(key)
                predictions_df = predictions_df.reset_index()
                predictions_df['date'] = pd.to_datetime(predictions_df['date'])
                predictions_df['permno'] = predictions_df['permno'].astype(int)
                predictions_by_n_factors[n_factors] = predictions_df
    return predictions_by_n_factors

# Load predictions for all models
all_predictions = {}
for base_path in base_paths:
    results_path = base_path / 'results' / 'asset_pricing' / 'all_predictions.h5'
    model_name = base_path.parts[-1].split('_')[-1]
    all_predictions[model_name] = load_predictions(str(results_path))

# Function to calculate value-weighted decile portfolios
def calculate_deciles(predicted, actual, mktcap_lag):
    deciles = pd.qcut(predicted.rank(method='first'), 10, labels=False) + 1
    top_decile = actual[deciles == 10]
    bottom_decile = actual[deciles == 1]
    top_weights = mktcap_lag[deciles == 10]
    bottom_weights = mktcap_lag[deciles == 1]

    # Normalize weights
    top_weights /= top_weights.sum()
    bottom_weights /= bottom_weights.sum()

    top_return = np.average(top_decile, weights=top_weights)
    bottom_return = np.average(bottom_decile, weights=bottom_weights)
    zero_net_investment_return = top_return - bottom_return
    return top_return, bottom_return, zero_net_investment_return

# Function to calculate additional performance metrics
def calculate_metrics(returns):
    avg_return = returns.mean()
    volatility = returns.std()
    var_95 = returns.quantile(0.05)
    turnover = (returns.diff().abs().sum()) / (2 * returns.abs().sum())
    return avg_return, volatility, var_95, turnover

# Function to evaluate economic performance for each model
def evaluate_performance(predictions_df, original_df):
    predictions_df = predictions_df.sort_values(by='date')
    merged_df = pd.merge(predictions_df, original_df[['date', 'permno', 'mktcap_lag']], on=['date', 'permno'])
    monthly_returns = merged_df.groupby('date').apply(lambda x: calculate_deciles(x['y_pred'], x['y_true'], x['mktcap_lag']))

    top_returns = monthly_returns.apply(lambda x: x[0])
    bottom_returns = monthly_returns.apply(lambda x: x[1])
    zero_net_investment_returns = monthly_returns.apply(lambda x: x[2])

    performance_df = pd.DataFrame({
        'Top_Decile_Returns': top_returns,
        'Bottom_Decile_Returns': bottom_returns,
        'Zero_Net_Investment_Returns': zero_net_investment_returns
    })
    performance_df['Cumulative_Top_Returns'] = (1 + performance_df['Top_Decile_Returns']).cumprod() - 1
    performance_df['Cumulative_Bottom_Returns'] = (1 + performance_df['Bottom_Decile_Returns']).cumprod() - 1
    performance_df['Cumulative_Zero_Net_Returns'] = (1 + performance_df['Zero_Net_Investment_Returns']).cumprod() - 1

    sharpe_ratio = performance_df['Zero_Net_Investment_Returns'].mean() / performance_df['Zero_Net_Investment_Returns'].std()

    avg_return, volatility, var_95, turnover = calculate_metrics(performance_df['Zero_Net_Investment_Returns'])

    return performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover

# Example S&P 500 permnos (This should be your actual list of S&P 500 permnos)
s_and_p_permnos = original_df['permno'].unique()  # Assuming all permnos are part of S&P 500 for this example

# Function to calculate S&P 500 returns
def calculate_sp500_returns(predictions_df, original_df, s_and_p_permnos):
    # Filter the dataframe for S&P 500 stocks
    sp500_df = original_df[original_df['permno'].isin(s_and_p_permnos)]

    # Merge predictions with S&P 500 stock data
    merged_df = pd.merge(predictions_df, sp500_df[['date', 'permno', 'mktcap_lag']], on=['date', 'permno'], how='inner')

    # Calculate value-weighted returns
    def weighted_return(df):
        weights = df['mktcap_lag'] / df['mktcap_lag'].sum()
        return np.sum(weights * df['y_true'])

    # Group by date to calculate the S&P 500 return for each month
    sp500_returns = merged_df.groupby('date').apply(weighted_return)

    return sp500_returns

# Process one set of predictions (example using model '0' and factor '1')
model = '0'
n_factors = '1.0'
predictions_df = all_predictions[model][n_factors]

# Evaluate performance for the model
performance_df, sharpe_ratio, avg_return, volatility, var_95, turnover = evaluate_performance(predictions_df, original_df)

# Calculate S&P 500 returns
sp500_returns = calculate_sp500_returns(predictions_df, original_df, s_and_p_permnos)
sp500_cumulative_returns = (1 + sp500_returns).cumprod() - 1

# Plot the results
plt.figure(figsize=(12, 8))
plt.plot(performance_df.index, performance_df['Cumulative_Top_Returns'], label='Top Decile Returns')
# plt.plot(performance_df.index, performance_df['Cumulative_Bottom_Returns'], label='Bottom Decile Returns')
# plt.plot(performance_df.index, performance_df['Cumulative_Zero_Net_Returns'], label=f'Model {model} (Sharpe: {sharpe_ratio:.2f})')
# plt.plot(sp500_cumulative_returns.index, sp500_cumulative_returns, label='S&P 500')
plt.title(f'Cumulative Returns for Model {model} and Factor {n_factors}')
plt.xlabel('Date')
plt.ylabel('Cumulative Returns')
plt.legend()
plt.show()

# Print metrics
print(f"Model {model}, Factor {n_factors}")
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
print(f"Average Return: {avg_return:.2%}")
print(f"Volatility: {volatility:.2%}")
print(f"Value-at-Risk (95%): {var_95:.2%}")
print(f"Turnover: {turnover:.2%}")


In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np

# Mount Google Drive
drive.mount('/content/drive')

# Define the base paths for each dataset
base_paths = [
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_2_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_1_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_0_ret_excess_fwd'
]

# Function to load all predictions from a specific HDF5 file
def load_predictions_from_hdf5(results_path):
    all_predictions = {}
    with pd.HDFStore(str(results_path / 'all_predictions.h5'), mode='r') as store:
        keys = store.keys()
        for key in keys:
            all_predictions[key] = store[key]
    return all_predictions

# Function to print all predictions for each n_factors
def print_all_predictions(all_predictions):
    for key, df in all_predictions.items():
        # Extract n_factors and l1_reg from the key
        parts = key.split('_')
        n_factors = parts[2]
        l1_reg = parts[-1]
        print(f"Predictions for n_factors={n_factors}, l1_reg={l1_reg}:")
        print(df.head())  # Display the first few rows of each DataFrame
        print()

# Iterate over each base path, load predictions, and print them
for base_path_str in base_paths:
    base_path = Path(base_path_str)
    results_path = base_path / 'results' / 'asset_pricing'
    print(f"Loading predictions from: {results_path}")
    all_predictions = load_predictions_from_hdf5(results_path)
    print_all_predictions(all_predictions)


In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np

# Mount Google Drive
drive.mount('/content/drive')

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3_ret_excess_fwd')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Load all predictions from the HDF5 file
all_predictions = {}
with pd.HDFStore(str(results_path / 'all_predictions.h5'), mode='r') as store:
    keys = store.keys()
    for key in keys:
        all_predictions[key] = store[key]

# Display the loaded predictions for inspection
for key, df in all_predictions.items():
    print(f"Predictions for {key}:")
    print(df.head())  # Display the first few rows of each DataFrame
    print()

# Function to print all predictions for each n_factors
def print_all_predictions(all_predictions):
    for key, df in all_predictions.items():
        # Extract n_factors from the key
        n_factors = key.split('_')[2]  # Adjust based on your key format
        l1_reg = key.split('_')[-1]  # Adjust based on your key format
        print(f"Predictions for n_factors={n_factors}, l1_reg={l1_reg}:")
        print(df.head())  # Display the first few rows of each DataFrame
        print()

# Call the function to print all predictions
print_all_predictions(all_predictions)


In [ ]:
from sklearn.metrics import r2_score
import pandas as pd
from pathlib import Path
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define the base paths for Google Drive
base_paths = [
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_0'),
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_1'),
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_2'),
    Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3')
]

# Initialize an empty dictionary to hold all predictions
all_predictions = {}

# Iterate over each base path
for base_path in base_paths:
    results_path = base_path / 'results' / 'asset_pricing'
    predictions_file = results_path / 'all_predictions.h5'

    # Check if the predictions file exists
    if predictions_file.exists():
        with pd.HDFStore(str(predictions_file), mode='r') as store:
            # Load all dataframes from the HDF5 file and store in the dictionary
            for key in store.keys():
                factor_l1_key = key.split('/')[-1]
                all_predictions[(base_path.name, factor_l1_key)] = store[key]
    else:
        print(f'Predictions file not found at: {predictions_file}')

# Function to calculate R^2 for each prediction DataFrame
def calculate_r2(predictions):
    r2_values = {}
    for (model, key), df in predictions.items():
        n_factors = key.split('_')[2]
        y_true = df['y_true'].astype(np.float64)
        y_pred = df['y_pred'].astype(np.float64)
        r2 = r2_score(y_true, y_pred)
        if model not in r2_values:
            r2_values[model] = {}
        r2_values[model][n_factors] = r2
    return r2_values

# Calculate R^2 values for all predictions
r2_dict = calculate_r2(all_predictions)

# Convert the dictionary to a DataFrame
r2_df = pd.DataFrame(r2_dict).T

# Convert the DataFrame to a LaTeX table
latex_r2 = r2_df.to_latex(index=True, header=True, float_format="%.2f")

# Print the LaTeX table
print("R^2 Values:")
print(latex_r2)


**Predictive R^2**

In [ ]:
from sklearn.metrics import r2_score
import pandas as pd
from pathlib import Path
from google.colab import drive
import numpy as np
# Mount Google Drive
drive.mount('/content/drive')

# Define the base paths for each dataset
base_paths = [
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_2_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_1_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_0_ret_excess_fwd'
]

# Function to load all predictions from a specific HDF5 file
def load_predictions_from_hdf5(results_path):
    all_predictions = {}
    with pd.HDFStore(str(results_path / 'all_predictions.h5'), mode='r') as store:
        keys = store.keys()
        for key in keys:
            all_predictions[key] = store[key]
    return all_predictions

# Function to calculate R^2 for each prediction DataFrame
def calculate_r2(predictions):
    r2_values = {}
    for key, df in predictions.items():
        n_factors = key.split('_')[2]
        y_true = df['y_true'].astype(np.float64)
        y_pred = df['y_pred'].astype(np.float64)
        r2 = r2_score(y_true, y_pred)
        r2_values[n_factors] = r2
    return r2_values

# Dictionary to store R^2 values for each model
r2_dict = {}

# Iterate over each base path, load predictions, and calculate R^2
for base_path_str in base_paths:
    base_path = Path(base_path_str)
    results_path = base_path / 'results' / 'asset_pricing'
    model_name = base_path.parts[-1]  # Use the full directory name as the model name
    print(f"Loading predictions from: {results_path}")
    all_predictions = load_predictions_from_hdf5(results_path)
    r2_values = calculate_r2(all_predictions)
    r2_dict[model_name] = r2_values

# Convert the dictionary to a DataFrame
r2_df = pd.DataFrame(r2_dict).T

# Convert the DataFrame to a LaTeX table
latex_r2 = r2_df.to_latex(index=True, header=True, float_format="%.2f")

# Print the LaTeX table
print("R^2 Values:")
print(latex_r2)


**Total R^2**

**Predictive $R^2$ for $x_t$**

In [ ]:
from sklearn.metrics import r2_score
import pandas as pd
from pathlib import Path
from google.colab import drive
import numpy as np
# Mount Google Drive
drive.mount('/content/drive')

# Define the base paths for each dataset
base_paths = [
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_2',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_1',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_0'
]

# Function to load all predictions from a specific HDF5 file
def load_predictions_from_hdf5(results_path):
    all_predictions = {}
    with pd.HDFStore(str(results_path / 'all_predictions.h5'), mode='r') as store:
        keys = store.keys()
        for key in keys:
            all_predictions[key] = store[key]
    return all_predictions

# Function to calculate R^2 for each prediction DataFrame
def calculate_r2(predictions):
    r2_values = {}
    for key, df in predictions.items():
        n_factors = key.split('_')[2]
        y_true = df['y_true'].astype(np.float64)
        y_pred = df['y_pred'].astype(np.float64)
        r2 = r2_score(y_true, y_pred)
        r2_values[n_factors] = r2
    return r2_values

# Dictionary to store R^2 values for each model
r2_dict = {}

# Iterate over each base path, load predictions, and calculate R^2
for base_path_str in base_paths:
    base_path = Path(base_path_str)
    results_path = base_path / 'results' / 'asset_pricing'
    model_name = base_path.parts[-1]  # Use the full directory name as the model name
    print(f"Loading predictions from: {results_path}")
    all_predictions = load_predictions_from_hdf5(results_path)
    r2_values = calculate_r2(all_predictions)
    r2_dict[model_name] = r2_values

# Convert the dictionary to a DataFrame
r2_df = pd.DataFrame(r2_dict).T

# Convert the DataFrame to a LaTeX table
latex_r2 = r2_df.to_latex(index=True, header=True, float_format="%.2f")

# Print the LaTeX table
print("R^2 Values:")
print(latex_r2)


**CALCULATION OF $R^2$ FOR $x_t$ : Retrieve original stock characteristics**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install pandas pyarrow

import pandas as pd

# Define the file path
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/stock_characteristics_monthly_py.parquet'

# Load the Parquet file into a DataFrame
df = pd.read_parquet(file_path, engine='pyarrow')

# Check the loaded data
print(df.head())

import pandas as pd

# Assuming df is your DataFrame

# Rename the 'month' column to 'date'
df = df.rename(columns={'month': 'date'})

# Ensure 'df' has a MultiIndex ['date', 'permno']
df = df.set_index(['date', 'permno'])

# Sort the MultiIndex
df = df.sort_index()


# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade"]
# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes
characteristics = [f'characteristic_{code}' for code in all_codes]


# # Filter for dates starting from '1993'
# df = df.loc[pd.IndexSlice['1960':, :], :]

# # Create forward returns
# df['fwd_ret_excess'] = df['ret_excess'].unstack('permno').shift(-1).stack(dropna=False)



# import pandas as pd
# import numpy as np
# from sklearn.preprocessing import quantile_transform, StandardScaler
# from tqdm import tqdm
# from joblib import Parallel, delayed

# # Define the function to apply quantile transformation for a single date
# def process_date(date, df, characteristics):
#     group = df.loc[date, characteristics]
#     transformed = quantile_transform(group, copy=True, n_quantiles=group.shape[0])
#     transformed_df = pd.DataFrame(transformed, columns=characteristics, index=group.index)
#     transformed_df = transformed_df.mul(2).sub(1)
#     transformed_df['date'] = date  # Add the date back to the DataFrame
#     return transformed_df

# # Initialize the progress bar
# dates = df.index.get_level_values('date').unique()
# pbar = tqdm(dates, desc="Processing Dates")

# # Apply the transformation with parallel processing
# normalized_data = Parallel(n_jobs=-1)(delayed(process_date)(date, df, characteristics) for date in pbar)

# # Concatenate all the normalized data back into a single DataFrame
# normalized_df = pd.concat(normalized_data)



# # Reset and set the index to match the original DataFrame's multi-index
# normalized_df = normalized_df.reset_index().set_index(['date', 'permno'])

# # Ensure the index order matches the original DataFrame
# normalized_df = normalized_df.reindex(df.index)

# # Assign the normalized values back to the original DataFrame
# df.loc[:, characteristics] = normalized_df[characteristics].values

# # Close the progress bar
# pbar.close()

# # Scale the variables
# scaler = StandardScaler()
# df[characteristics] = scaler.fit_transform(df[characteristics])

# # Clip the returns columns
# df.loc[:, ['ret_excess', 'fwd_ret_excess']] = df.loc[:, ['ret_excess', 'fwd_ret_excess']].clip(lower=-1, upper=1.0)


# Print the first few rows of the scaled DataFrame to verify
print(df.head())

# Add 'ret_excess' to the list of columns
columns_to_select = characteristics

# Select these columns from the DataFrame
characteristics_data = df[columns_to_select]

# Define a function to fill missing values with the median for each group
def fill_missing_with_median(group):
    return group.fillna(group.median())

# Apply the function to each group (each date) and preserve the index
df_filled = characteristics_data.groupby(level='date').apply(fill_missing_with_median)

# Remove the additional 'date' level in the index introduced by groupby
df_filled.index = df_filled.index.droplevel(0)

# Add the filled columns back to the original DataFrame
df.update(df_filled)

print(df)

# Display the updated DataFrame to check the results
print(df.head())



In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd

# Mount Google Drive
drive.mount('/content/drive')

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3')




import pandas as pd

# # Define the file path
# file_path = '/content/drive/My Drive/Tidy Finance ML dataset/stock_characteristics_monthly_py.parquet'

# # Load the Parquet file into a DataFrame
# df = pd.read_parquet(file_path, engine='pyarrow')

# file_path = '/content/drive/My Drive/Tidy Finance ML dataset/beta.parquet'

# # Make sure merged_df1 is a DataFrame and not a function or any other type.
# df = pd.read_parquet(file_path)

# # Remove rows where -2 is in any column
# df = df[~(df == -2).any(axis=1)]


# # Rename the 'month' column to 'date'
# df = df.rename(columns={'month': 'date'})

# # Ensure 'df' has a MultiIndex ['date', 'permno']
# df = df.set_index(['date', 'permno'])

# # Sort the MultiIndex
# df = df.sort_index()


# Check the loaded data
print(df.head())


# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Load a specific prediction from the HDF5 file
key_to_load = '/n_factors_1.0_l1_reg_1e-09'  # Replace with the key of the DataFrame you want to load
with pd.HDFStore(str(results_path / 'all_predictions.h5'), mode='r') as store:
    specific_prediction = store[key_to_load]

# Display the loaded prediction for inspection
print(f"Predictions for {key_to_load}:")
print(specific_prediction.head())  # Display the first few rows of the DataFrame

# # Ensure 'df' has a MultiIndex ['date', 'permno']
# specific_prediction = specific_prediction.set_index(['date', 'permno'])

# Sort the MultiIndex
specific_prediction = specific_prediction.sort_index()

# Inner merge on MultiIndex
merged_df = pd.merge(specific_prediction, df, left_index=True, right_index=True, how='inner')

# Display the merged DataFrame
print(merged_df)


import pandas as pd
import numpy as np


############################
## CALCULATE XT FOR TPRED ##
############################

# Function to calculate x_t for a given month using Ridge Regression
def calculate_xt_y_pred(df, date, alpha=0):
    # Filter DataFrame for the given date
    df_month = df.loc[date]

    # Separate returns and characteristics
    r_t = df_month['y_pred'].values
    characteristic_columns = df_month.drop(columns=['y_pred']).columns
    Z_t_minus_1 = df_month.drop(columns=['y_pred']).values

    # Calculate (Z_{t-1}'Z_{t-1})^{-1}Z_{t-1}r_t using Ridge Regression
    Z_t_minus_1_T = Z_t_minus_1.T
    Z_t_minus_1_cov = Z_t_minus_1_T @ Z_t_minus_1
    Z_t_minus_1_cov_reg = Z_t_minus_1_cov + alpha * np.identity(Z_t_minus_1_cov.shape[0])

    Z_t_minus_1_cov_inv = np.linalg.inv(Z_t_minus_1_cov_reg)

    # Calculate x_t
    x_t = Z_t_minus_1_cov_inv @ Z_t_minus_1_T @ r_t

    return x_t, characteristic_columns

# Calculate x_t for all unique dates in the dataset
def calculate_xt_all_dates_pred(df, alpha=1e-6):
    xt_results = {}
    characteristic_columns = None

    unique_dates = df.index.get_level_values('date').unique()
    for date in unique_dates:
        x_t, characteristic_columns = calculate_xt_y_pred(df, date, alpha)
        xt_results[date] = x_t

    # Convert dictionary to DataFrame
    xt_df = pd.DataFrame.from_dict(xt_results, orient='index')

    # Assign original characteristic names as column names
    if characteristic_columns is not None:
        xt_df.columns = characteristic_columns

    # Preserve the index name
    xt_df.index.name = 'date'

    return xt_df

############################
## CALCULATE XT FOR YTRUE ##
############################


def calculate_xt_y_true(df, date, alpha=0):
    # Filter DataFrame for the given date
    df_month = df.loc[date]

    # Separate returns and characteristics
    r_t = df_month['y_true'].values
    characteristic_columns = df_month.drop(columns=['y_true']).columns
    Z_t_minus_1 = df_month.drop(columns=['y_true']).values

    # Calculate (Z_{t-1}'Z_{t-1})^{-1}Z_{t-1}r_t using Ridge Regression
    Z_t_minus_1_T = Z_t_minus_1.T
    Z_t_minus_1_cov = Z_t_minus_1_T @ Z_t_minus_1
    Z_t_minus_1_cov_reg = Z_t_minus_1_cov + alpha * np.identity(Z_t_minus_1_cov.shape[0])

    Z_t_minus_1_cov_inv = np.linalg.inv(Z_t_minus_1_cov_reg)

    # Calculate x_t
    x_t = Z_t_minus_1_cov_inv @ Z_t_minus_1_T @ r_t

    return x_t, characteristic_columns

# Calculate x_t for all unique dates in the dataset
def calculate_xt_all_dates_true(df, alpha=1e-04):
    xt_results = {}
    characteristic_columns = None

    unique_dates = df.index.get_level_values('date').unique()
    for date in unique_dates:
        x_t, characteristic_columns = calculate_xt_y_true(df, date, alpha)
        xt_results[date] = x_t

    # Convert dictionary to DataFrame
    xt_df = pd.DataFrame.from_dict(xt_results, orient='index')

    # Assign original characteristic names as column names
    if characteristic_columns is not None:
        xt_df.columns = characteristic_columns

    # Preserve the index name
    xt_df.index.name = 'date'

    return xt_df


# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade"]
# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes
characteristics = [f'characteristic_{code}' for code in all_codes]


columns_to_select = characteristics + ["y_pred"]
characteristics_data_pred = merged_df[columns_to_select]

columns_to_select = characteristics + ["y_true"]
characteristics_data_true = merged_df[columns_to_select]


# Example usage
xt_true = calculate_xt_all_dates_true(characteristics_data_true)


xt_pred = calculate_xt_all_dates_pred(characteristics_data_pred)


# # Print the results for the first few dates to confirm
# print(xt_pred.head(),xt_true.head())

import pandas as pd
import numpy as np

# Assuming xt_pred and xt_true are DataFrames with the same shape and index

# Step 1: Calculate the residuals
residuals = xt_true - xt_pred

# Step 2: Compute the sum of squared residuals (SSR)
ssr = np.sum(np.square(residuals.values))

# Step 3: Compute the total sum of squares (TSS)
tss = np.sum(np.square(xt_true.values - xt_true.mean().values))

# Step 4: Calculate R^2
r_squared = 1 - (ssr / tss)

print("R^2 for x_t:", r_squared)

def calculate_r2(xt_true, xt_pred):
    residuals = xt_true - xt_pred
    ssr = np.sum(np.square(residuals.values))
    tss = np.sum(np.square(xt_true.values - xt_true.mean().values))
    r_squared = 1 - (ssr / tss)
    return r_squared






In [ ]:
print(columns_to_select)
print(merged_df.columns)

In [ ]:
print(df.index)
print(specific_prediction.index)

In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np


# Define the base paths for each dataset
base_paths = [
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_2_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_1_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_0_ret_excess_fwd'
]


# Mount Google Drive
drive.mount('/content/drive')

# # Define the base path for Google Drive
# base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3_ret_excess_fwd')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Load all predictions from the HDF5 file

def load_model_predictions(results_path):
    all_predictions = {}
    with pd.HDFStore(str(results_path / 'all_predictions.h5'), mode='r') as store:
        keys = store.keys()
        for key in keys:
            all_predictions[key] = store[key]
    return all_predictions

def calculate_xt_for_model(all_predictions, characteristic_df, hidden_layers):
    r2_df = pd.DataFrame()  # Initialize an empty DataFrame
    for i, (key, df_pred) in enumerate(all_predictions.items(), start=1):
        print(key)
        df_pred = df_pred.sort_index()
        # Inner merge on MultiIndex
        merged_df = pd.merge(df_pred, characteristic_df, left_index=True, right_index=True, how='inner')

        columns_to_select = characteristics + ["y_pred"]
        characteristics_data_pred = merged_df[columns_to_select]

        columns_to_select = characteristics + ["y_true"]
        characteristics_data_true = merged_df[columns_to_select]

        # Example usage
        xt_true = calculate_xt_all_dates_true(characteristics_data_true)

        xt_pred = calculate_xt_all_dates_pred(characteristics_data_pred)

        xt_r2 = calculate_r2(xt_true, xt_pred)

        print(f"$R^2$ for $x_t$ for {key}: {xt_r2}")

        # Add the R^2 value as a new column to the DataFrame
        r2_df[f'{i}'] = [xt_r2]

    # Set the row name
    r2_df.index = [f'$CA_{hidden_layers}$']

    return r2_df

# Call the function
all_predictions = load_model_predictions(results_path)
r2_model = calculate_xt_for_model(all_predictions, df, 1)

models = []
for path in base_paths:
    results_path = path / 'results' / 'asset_pricing'
    all_predictions = load_model_predictions(results_path)
    r2_model = calculate_xt_for_model(all_predictions, df, hidden_layers)
    models_df = pd.concat([models_df, r2_model])
    models.append(r2_model)
    models_df = pd.concat(models)

print(models_df)


**CALCULATION OF $R^2$ FOR $x_t$ : Define functons for calculation of $x_t$**

In [ ]:

############################
## CALCULATE XT FOR TPRED ##
############################

# Function to calculate x_t for a given month using Ridge Regression
def calculate_xt_y_pred(df, date, alpha=0):
    # Filter DataFrame for the given date
    df_month = df.loc[date]

    # Separate returns and characteristics
    r_t = df_month['y_pred'].values
    characteristic_columns = df_month.drop(columns=['y_pred']).columns
    Z_t_minus_1 = df_month.drop(columns=['y_pred']).values

    # Calculate (Z_{t-1}'Z_{t-1})^{-1}Z_{t-1}r_t using Ridge Regression
    Z_t_minus_1_T = Z_t_minus_1.T
    Z_t_minus_1_cov = Z_t_minus_1_T @ Z_t_minus_1
    Z_t_minus_1_cov_reg = Z_t_minus_1_cov + alpha * np.identity(Z_t_minus_1_cov.shape[0])

    Z_t_minus_1_cov_inv = np.linalg.inv(Z_t_minus_1_cov_reg)

    # Calculate x_t
    x_t = Z_t_minus_1_cov_inv @ Z_t_minus_1_T @ r_t

    return x_t, characteristic_columns

# Calculate x_t for all unique dates in the dataset
def calculate_xt_all_dates_pred(df, alpha=1e-6):
    xt_results = {}
    characteristic_columns = None

    unique_dates = df.index.get_level_values('date').unique()
    for date in unique_dates:
        x_t, characteristic_columns = calculate_xt_y_pred(df, date, alpha)
        xt_results[date] = x_t

    # Convert dictionary to DataFrame
    xt_df = pd.DataFrame.from_dict(xt_results, orient='index')

    # Assign original characteristic names as column names
    if characteristic_columns is not None:
        xt_df.columns = characteristic_columns

    # Preserve the index name
    xt_df.index.name = 'date'

    return xt_df

############################
## CALCULATE XT FOR YTRUE ##
############################


def calculate_xt_y_true(df, date, alpha=0):
    # Filter DataFrame for the given date
    df_month = df.loc[date]

    # Separate returns and characteristics
    r_t = df_month['y_true'].values
    characteristic_columns = df_month.drop(columns=['y_true']).columns
    Z_t_minus_1 = df_month.drop(columns=['y_true']).values

    # Calculate (Z_{t-1}'Z_{t-1})^{-1}Z_{t-1}r_t using Ridge Regression
    Z_t_minus_1_T = Z_t_minus_1.T
    Z_t_minus_1_cov = Z_t_minus_1_T @ Z_t_minus_1
    Z_t_minus_1_cov_reg = Z_t_minus_1_cov + alpha * np.identity(Z_t_minus_1_cov.shape[0])

    Z_t_minus_1_cov_inv = np.linalg.inv(Z_t_minus_1_cov_reg)

    # Calculate x_t
    x_t = Z_t_minus_1_cov_inv @ Z_t_minus_1_T @ r_t

    return x_t, characteristic_columns

# Calculate x_t for all unique dates in the dataset
def calculate_xt_all_dates_true(df, alpha=1e-6):
    xt_results = {}
    characteristic_columns = None

    unique_dates = df.index.get_level_values('date').unique()
    for date in unique_dates:
        x_t, characteristic_columns = calculate_xt_y_true(df, date, alpha)
        xt_results[date] = x_t

    # Convert dictionary to DataFrame
    xt_df = pd.DataFrame.from_dict(xt_results, orient='index')

    # Assign original characteristic names as column names
    if characteristic_columns is not None:
        xt_df.columns = characteristic_columns

    # Preserve the index name
    xt_df.index.name = 'date'

    return xt_df

########################
## CALCULATION OF R^2 ##
########################

# def calculate_r2(xt_true, xt_pred):
#     residuals = xt_true - xt_pred
#     ssr = np.sum(np.square(residuals.values))
#     tss = np.sum(np.square(xt_true.values - xt_true.mean().values))
#     r_squared = 1 - (ssr / tss)
#     return r_squared

import numpy as np

def calculate_r2(xt_true, xt_pred):
    # Flatten the dataframes to create a single list of all values
    y_true = xt_true.values.flatten()
    y_pred = xt_pred.values.flatten()

    # Calculate residuals
    residuals = y_true - y_pred

    # Calculate the sum of squares of residuals
    ssr = np.sum(np.square(residuals))

    # Calculate the total sum of squares
    tss = np.sum(np.square(y_true - np.mean(y_true)))

    # Calculate R^2
    r_squared = 1 - (ssr / tss)

    return r_squared



**

**CALCULATION OF $R^2$ FOR $x_t$ : FORWARD RETURNS $R^2$ FOR $x_t$**

In [ ]:

##################################################################
# Define the base paths for each dataset
base_paths = [
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_2_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_1_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_0_ret_excess_fwd'
]

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Define the hidden layers corresponding to each base path
hidden_layers_list = [3, 2, 1, 0]

def load_model_predictions(results_path):
    all_predictions = {}
    with pd.HDFStore(str(results_path / 'all_predictions.h5'), mode='r') as store:
        keys = store.keys()
        for key in keys:
            all_predictions[key] = store[key]
    return all_predictions

def calculate_xt_for_model(all_predictions, characteristic_df, hidden_layer):
    r2_df = pd.DataFrame()  # Initialize an empty DataFrame
    for i, (key, df_pred) in enumerate(all_predictions.items(), start=1):
        print(key)
        df_pred = df_pred.sort_index()
        # Inner merge on MultiIndex
        merged_df = pd.merge(df_pred, characteristic_df, left_index=True, right_index=True, how='inner')

        columns_to_select = characteristics + ["y_pred"]
        characteristics_data_pred = merged_df[columns_to_select]

        columns_to_select = characteristics + ["y_true"]
        characteristics_data_true = merged_df[columns_to_select]

        # Example usage
        xt_true = calculate_xt_all_dates_true(characteristics_data_true)
        # print(xt_true)
        xt_pred = calculate_xt_all_dates_pred(characteristics_data_pred)
        print(type(xt_pred))
        xt_r2 = calculate_r2(xt_true, xt_pred)

        print(f"$R^2$ for $x_t$ for {key}: {xt_r2}")

        # Add the R^2 value as a new column to the DataFrame
        r2_df[f'{i}'] = [xt_r2]

    # Set the row name
    r2_df.index = [f'$CA_{hidden_layer}$']

    return r2_df

models = []
for path, hidden_layer in zip(base_paths, hidden_layers_list):
    results_path = Path(path) / 'results' / 'asset_pricing'
    all_predictions = load_model_predictions(results_path)
    r2_model = calculate_xt_for_model(all_predictions, df, hidden_layer)
    models.append(r2_model)

# Concatenate all DataFrames in the models list by rows
models_df = pd.concat(models, axis=0)


In [ ]:
df_pred

**CALCULATION OF $R^2$ FOR $x_t$ : FORWARD RETURNS $R^2$ FOR $x_t$**: Contemporary returns**

In [ ]:

# Define the base paths for different models
base_paths = [
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_0',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_1',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_2',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3'
]


##################################################################
# Define the base paths for each dataset

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Define the hidden layers corresponding to each base path
hidden_layers_list = [3, 2, 1, 0]

def load_model_predictions(results_path):
    all_predictions = {}
    with pd.HDFStore(str(results_path / 'all_predictions.h5'), mode='r') as store:
        keys = store.keys()
        for key in keys:
            all_predictions[key] = store[key]
    return all_predictions

def calculate_xt_for_model(all_predictions, characteristic_df, hidden_layer):
    r2_df = pd.DataFrame()  # Initialize an empty DataFrame
    for i, (key, df_pred) in enumerate(all_predictions.items(), start=1):
        print(key)
        df_pred = df_pred.sort_index()
        # Inner merge on MultiIndex
        merged_df = pd.merge(df_pred, characteristic_df, left_index=True, right_index=True, how='inner')

        columns_to_select = characteristics + ["y_pred"]
        characteristics_data_pred = merged_df[columns_to_select]

        columns_to_select = characteristics + ["y_true"]
        characteristics_data_true = merged_df[columns_to_select]

        # Example usage
        xt_true = calculate_xt_all_dates_true(characteristics_data_true)
        # print(xt_true)
        xt_pred = calculate_xt_all_dates_pred(characteristics_data_pred)
        print(type(xt_pred))
        xt_r2 = calculate_r2(xt_true, xt_pred)

        print(f"$R^2$ for $x_t$ for {key}: {xt_r2}")

        # Add the R^2 value as a new column to the DataFrame
        r2_df[f'{i}'] = [xt_r2]

    # Set the row name
    r2_df.index = [f'$CA_{hidden_layer}$']

    return r2_df

models = []
for path, hidden_layer in zip(base_paths, hidden_layers_list):
    results_path = Path(path) / 'results' / 'asset_pricing'
    all_predictions = load_model_predictions(results_path)
    r2_model = calculate_xt_for_model(all_predictions, df, hidden_layer)
    models.append(r2_model)

# Concatenate all DataFrames in the models list by rows
models_df = pd.concat(models, axis=0)


In [ ]:
# Generate the LaTeX table from the DataFrame
latex_table = models_df.to_latex(escape=False)

# Print the LaTeX table
print(latex_table)


**CHECK**

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
!pip install pandas pyarrow

import pandas as pd
import numpy as np
from sklearn.preprocessing import quantile_transform, StandardScaler
from tqdm import tqdm
from joblib import Parallel, delayed
from pathlib import Path

# Define the file path
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/stock_characteristics_monthly_py.parquet'

# Load the Parquet file into a DataFrame
df = pd.read_parquet(file_path, engine='pyarrow')

# Check the loaded data
print(df.head())

# Rename and set index
df = df.rename(columns={'month': 'date'}).set_index(['date', 'permno']).sort_index()


# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade"]
# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes
characteristics = [f'characteristic_{code}' for code in all_codes]

# Filter dates
df = df.loc[pd.IndexSlice['1960':, :], :]

# Create forward returns
df['fwd_ret_excess'] = df['ret_excess'].unstack('permno').shift(-1).stack(dropna=False)

# Quantile transform characteristics
def process_date(date, df, characteristics):
    group = df.loc[date, characteristics]
    transformed = quantile_transform(group, copy=True, n_quantiles=group.shape[0])
    transformed_df = pd.DataFrame(transformed, columns=characteristics, index=group.index)
    transformed_df = transformed_df.mul(2).sub(1)
    transformed_df['date'] = date
    return transformed_df

dates = df.index.get_level_values('date').unique()
pbar = tqdm(dates, desc="Processing Dates")
normalized_data = Parallel(n_jobs=-1)(delayed(process_date)(date, df, characteristics) for date in pbar)
normalized_df = pd.concat(normalized_data).reset_index().set_index(['date', 'permno']).reindex(df.index)
df.loc[:, characteristics] = normalized_df[characteristics].values
pbar.close()

# Scale the variables
scaler = StandardScaler()
df[characteristics] = scaler.fit_transform(df[characteristics])

# # Clip returns
# df.loc[:, ['ret_excess', 'fwd_ret_excess']] = df.loc[:, ['ret_excess', 'fwd_ret_excess']].clip(lower=-1, upper=1.0)

# Fill missing values with median
def fill_missing_with_median(group):
    return group.fillna(group.median())

df_filled = df[characteristics].groupby(level='date').apply(fill_missing_with_median)
df_filled.index = df_filled.index.droplevel(0)
df.update(df_filled)

# Check results
print(df.head())

# Calculate x_t for predicted returns
def calculate_xt_y_pred(df, date, alpha=0):
    df_month = df.loc[date]
    r_t = df_month['y_pred'].values
    Z_t_minus_1 = df_month.drop(columns=['y_pred']).values
    Z_t_minus_1_T = Z_t_minus_1.T
    Z_t_minus_1_cov = Z_t_minus_1_T @ Z_t_minus_1
    Z_t_minus_1_cov_reg = Z_t_minus_1_cov + alpha * np.identity(Z_t_minus_1_cov.shape[0])
    Z_t_minus_1_cov_inv = np.linalg.inv(Z_t_minus_1_cov_reg)
    x_t = Z_t_minus_1_cov_inv @ Z_t_minus_1_T @ r_t
    return x_t

def calculate_xt_all_dates_pred(df, alpha=1e-6):
    xt_results = {}
    unique_dates = df.index.get_level_values('date').unique()
    for date in unique_dates:
        x_t = calculate_xt_y_pred(df, date, alpha)
        xt_results[date] = x_t
    xt_df = pd.DataFrame.from_dict(xt_results, orient='index')
    xt_df.index.name = 'date'
    return xt_df

# Calculate x_t for true returns
def calculate_xt_y_true(df, date, alpha=0):
    df_month = df.loc[date]
    r_t = df_month['y_true'].values
    Z_t_minus_1 = df_month.drop(columns=['y_true']).values
    Z_t_minus_1_T = Z_t_minus_1.T
    Z_t_minus_1_cov = Z_t_minus_1_T @ Z_t_minus_1
    Z_t_minus_1_cov_reg = Z_t_minus_1_cov + alpha * np.identity(Z_t_minus_1_cov.shape[0])
    Z_t_minus_1_cov_inv = np.linalg.inv(Z_t_minus_1_cov_reg)
    x_t = Z_t_minus_1_cov_inv @ Z_t_minus_1_T @ r_t
    return x_t

def calculate_xt_all_dates_true(df, alpha=1e-6):
    xt_results = {}
    unique_dates = df.index.get_level_values('date').unique()
    for date in unique_dates:
        x_t = calculate_xt_y_true(df, date, alpha)
        xt_results[date] = x_t
    xt_df = pd.DataFrame.from_dict(xt_results, orient='index')
    xt_df.index.name = 'date'
    return xt_df


def calculate_r2(xt_true, xt_pred):
    y_true = xt_true.values.flatten()
    y_pred = xt_pred.values.flatten()
    residuals = y_true - y_pred
    ssr = np.sum(np.square(residuals))
    tss = np.sum(np.square(y_true - np.mean(y_true)))
    r_squared = 1 - (ssr / tss)
    return r_squared

# Calculate R^2 for different models
def load_model_predictions(results_path):
    all_predictions = {}
    with pd.HDFStore(str(results_path / 'all_predictions.h5'), mode='r') as store:
        keys = store.keys()
        for key in keys:
            all_predictions[key] = store[key]
    return all_predictions

def calculate_xt_for_model(all_predictions, characteristic_df, hidden_layer):
    r2_df = pd.DataFrame()
    for i, (key, df_pred) in enumerate(all_predictions.items(), start=1):
        print(key)
        df_pred = df_pred.sort_index()
        merged_df = pd.merge(df_pred, characteristic_df, left_index=True, right_index=True, how='inner')
        characteristics_data_pred = merged_df[characteristics + ["y_pred"]]
        characteristics_data_true = merged_df[characteristics + ["y_true"]]
        xt_true = calculate_xt_all_dates_true(characteristics_data_true)
        xt_pred = calculate_xt_all_dates_pred(characteristics_data_pred)
        xt_r2 = calculate_r2(xt_true, xt_pred)
        print(f"$R^2$ for $x_t$ for {key}: {xt_r2}")
        r2_df[f'{i}'] = [xt_r2]
    r2_df.index = [f'$CA_{hidden_layer}$']
    return r2_df

base_paths = [
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_0',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_1',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_2',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3'
]

hidden_layers_list = [0, 1, 2, 3]
models = []
for path, hidden_layer in zip(base_paths, hidden_layers_list):
    results_path = Path(path) / 'results' / 'asset_pricing'
    all_predictions = load_model_predictions(results_path)
    r2_model = calculate_xt_for_model(all_predictions, df, hidden_layer)
    models.append(r2_model)

models_df = pd.concat(models, axis=0)
print(models_df)



In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.preprocessing import quantile_transform, StandardScaler
from joblib import Parallel, delayed
from pathlib import Path

# Define the file path
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/stock_characteristics_monthly_py.parquet'

# Load the Parquet file into a DataFrame
df = pd.read_parquet(file_path, engine='pyarrow')

# Rename and set index
df = df.rename(columns={'month': 'date'}).set_index(['date', 'permno']).sort_index()

# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade"]
# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes
characteristics = [f'characteristic_{code}' for code in all_codes]

# Filter dates
df = df.loc[pd.IndexSlice['1960':, :], :]

# Create forward returns
df['fwd_ret_excess'] = df['ret_excess'].unstack('permno').shift(-1).stack(dropna=False)

# Quantile transform characteristics
def process_date(date, df, characteristics):
    group = df.loc[date, characteristics]
    transformed = quantile_transform(group, copy=True, n_quantiles=group.shape[0])
    transformed_df = pd.DataFrame(transformed, columns=characteristics, index=group.index)
    transformed_df = transformed_df.mul(2).sub(1)
    transformed_df['date'] = date
    return transformed_df

dates = df.index.get_level_values('date').unique()
pbar = tqdm(dates, desc="Processing Dates")
normalized_data = Parallel(n_jobs=-1)(delayed(process_date)(date, df, characteristics) for date in pbar)
normalized_df = pd.concat(normalized_data).reset_index().set_index(['date', 'permno']).reindex(df.index)
df.loc[:, characteristics] = normalized_df[characteristics].values
pbar.close()

# Scale the variables
scaler = StandardScaler()
df[characteristics] = scaler.fit_transform(df[characteristics])

# Fill missing values with median
def fill_missing_with_median(group):
    return group.fillna(group.median())

df_filled = df[characteristics].groupby(level='date').apply(fill_missing_with_median)
df_filled.index = df_filled.index.droplevel(0)
df.update(df_filled)

# Check results
print(df.head())

# Calculate long-short portfolio returns for predicted and true returns
def calculate_long_short_portfolios(df, y_col):
    char_managed_portfolios = pd.DataFrame(index=df.index.get_level_values(0).unique(), columns=characteristics)
    for characteristic in tqdm(characteristics):
        # Rank the stocks for each date based on the characteristic
        df[f'rank_{characteristic}'] = df.groupby(level='date')[characteristic].rank(ascending=False)

        # Define the top and bottom deciles
        def top_decile(x):
            return x <= x.quantile(0.1)

        def bottom_decile(x):
            return x >= x.quantile(0.9)

        df[f'top_decile_{characteristic}'] = df.groupby(level='date')[f'rank_{characteristic}'].transform(top_decile)
        df[f'bottom_decile_{characteristic}'] = df.groupby(level='date')[f'rank_{characteristic}'].transform(bottom_decile)

        # Calculate the returns for the long-short portfolio
        top_decile_mask = df[f'top_decile_{characteristic}']
        bottom_decile_mask = df[f'bottom_decile_{characteristic}']

        long_returns = df.loc[top_decile_mask, y_col]
        short_returns = df.loc[bottom_decile_mask, y_col]

        long_short_returns = long_returns.groupby(level='date').mean() - short_returns.groupby(level='date').mean()

        # Store the results in the DataFrame
        char_managed_portfolios[characteristic] = long_short_returns

    # Replace NaN values with 0 if there are any
    char_managed_portfolios.fillna(0, inplace=True)

    return char_managed_portfolios

# Calculate x_t for predicted returns
def calculate_xt_all_dates_pred(df):
    return calculate_long_short_portfolios(df, 'y_pred')

# Calculate x_t for true returns
def calculate_xt_all_dates_true(df):
    return calculate_long_short_portfolios(df, 'y_true')

# Function to calculate R^2 for each portfolio and average them
def calculate_r2_per_portfolio(xt_true, xt_pred):
    r2_values = []
    for col in xt_true.columns:
        y_true = xt_true[col].values
        y_pred = xt_pred[col].values
        residuals = y_true - y_pred
        ssr = np.sum(np.square(residuals))
        tss = np.sum(np.square(y_true - np.mean(y_true)))
        r_squared = 1 - (ssr / tss)
        r2_values.append(r_squared)
    return np.mean(r2_values)

# Function to calculate R^2 for each model
def calculate_xt_for_model(all_predictions, characteristic_df, hidden_layer):
    r2_df = pd.DataFrame()
    for i, (key, df_pred) in enumerate(all_predictions.items(), start=1):
        print(key)
        df_pred = df_pred.sort_index()
        merged_df = pd.merge(df_pred, characteristic_df, left_index=True, right_index=True, how='inner')
        characteristics_data_pred = merged_df[characteristics + ["y_pred"]]
        characteristics_data_true = merged_df[characteristics + ["y_true"]]
        xt_true = calculate_xt_all_dates_true(characteristics_data_true)
        xt_pred = calculate_xt_all_dates_pred(characteristics_data_pred)
        xt_r2 = calculate_r2_per_portfolio(xt_true, xt_pred)
        print(f"$R^2$ for $x_t$ for {key}: {xt_r2}")
        r2_df[f'{i}'] = [xt_r2]
    r2_df.index = [f'$CA_{hidden_layer}$']
    return r2_df

base_paths = [
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_0',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_1',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_2',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3'
]

hidden_layers_list = [0, 1, 2, 3]
models = []
for path, hidden_layer in zip(base_paths, hidden_layers_list):
    results_path = Path(path) / 'results' / 'asset_pricing'
    all_predictions = load_model_predictions(results_path)
    r2_model = calculate_xt_for_model(all_predictions, df, hidden_layer)
    models.append(r2_model)

models_df = pd.concat(models, axis=0)
print(models_df)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import quantile_transform, StandardScaler
from tqdm import tqdm
from joblib import Parallel, delayed
from pathlib import Path

# Function to load model predictions
def load_model_predictions(results_path):
    all_predictions = {}
    with pd.HDFStore(str(results_path / 'all_predictions.h5'), mode='r') as store:
        keys = store.keys()
        for key in keys:
            all_predictions[key] = store[key]
    return all_predictions

# Function to calculate x_t for a given month using Ridge Regression for predicted returns
def calculate_xt_y_pred(df, date, alpha=0):
    df_month = df.loc[date]
    r_t = df_month['y_pred'].values
    Z_t_minus_1 = df_month.drop(columns=['y_pred']).values
    Z_t_minus_1_T = Z_t_minus_1.T
    Z_t_minus_1_cov = Z_t_minus_1_T @ Z_t_minus_1
    Z_t_minus_1_cov_reg = Z_t_minus_1_cov + alpha * np.identity(Z_t_minus_1_cov.shape[0])
    Z_t_minus_1_cov_inv = np.linalg.inv(Z_t_minus_1_cov_reg)
    x_t = Z_t_minus_1_cov_inv @ Z_t_minus_1_T @ r_t
    return x_t

# Function to calculate x_t for all dates for predicted returns
def calculate_xt_all_dates_pred(df, alpha=1e-6):
    xt_results = {}
    unique_dates = df.index.get_level_values('date').unique()
    for date in unique_dates:
        x_t = calculate_xt_y_pred(df, date, alpha)
        xt_results[date] = x_t
    xt_df = pd.DataFrame.from_dict(xt_results, orient='index')
    xt_df.index.name = 'date'
    return xt_df

# Function to calculate x_t for a given month using Ridge Regression for true returns
def calculate_xt_y_true(df, date, alpha=0):
    df_month = df.loc[date]
    r_t = df_month['y_true'].values
    Z_t_minus_1 = df_month.drop(columns=['y_true']).values
    Z_t_minus_1_T = Z_t_minus_1.T
    Z_t_minus_1_cov = Z_t_minus_1_T @ Z_t_minus_1
    Z_t_minus_1_cov_reg = Z_t_minus_1_cov + alpha * np.identity(Z_t_minus_1_cov.shape[0])
    Z_t_minus_1_cov_inv = np.linalg.inv(Z_t_minus_1_cov_reg)
    x_t = Z_t_minus_1_cov_inv @ Z_t_minus_1_T @ r_t
    return x_t

# Function to calculate x_t for all dates for true returns
def calculate_xt_all_dates_true(df, alpha=1e-6):
    xt_results = {}
    unique_dates = df.index.get_level_values('date').unique()
    for date in unique_dates:
        x_t = calculate_xt_y_true(df, date, alpha)
        xt_results[date] = x_t
    xt_df = pd.DataFrame.from_dict(xt_results, orient='index')
    xt_df.index.name = 'date'
    return xt_df

# Function to calculate R^2
def calculate_r2(xt_true, xt_pred):
    y_true = xt_true.values.flatten()
    y_pred = xt_pred.values.flatten()
    residuals = y_true - y_pred
    ssr = np.sum(np.square(residuals))
    tss = np.sum(np.square(y_true - np.mean(y_true)))
    r_squared = 1 - (ssr / tss)
    return r_squared

# Function to calculate R^2 for each model
def calculate_xt_for_model(all_predictions, characteristic_df, hidden_layer):
    r2_df = pd.DataFrame()
    for i, (key, df_pred) in enumerate(all_predictions.items(), start=1):
        print(key)
        df_pred = df_pred.sort_index()
        merged_df = pd.merge(df_pred, characteristic_df, left_index=True, right_index=True, how='inner')
        characteristics_data_pred = merged_df[characteristics + ["y_pred"]]
        characteristics_data_true = merged_df[characteristics + ["y_true"]]
        xt_true = calculate_xt_all_dates_true(characteristics_data_true)
        xt_pred = calculate_xt_all_dates_pred(characteristics_data_pred)
        xt_r2 = calculate_r2(xt_true, xt_pred)
        print(f"$R^2$ for $x_t$ for {key}: {xt_r2}")
        r2_df[f'{i}'] = [xt_r2]
    r2_df.index = [f'$CA_{hidden_layer}$']
    return r2_df

# Define base paths for different models
base_paths = [
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_0',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_1',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_2',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3'
]

models = []
hidden_layers_list = [3, 2, 1, 0]

for path, hidden_layer in zip(base_paths, hidden_layers_list):
    results_path = Path(path) / 'results' / 'asset_pricing'
    all_predictions = load_model_predictions(results_path)
    r2_model = calculate_xt_for_model(all_predictions, df, hidden_layer)
    models.append(r2_model)

models_df = pd.concat(models, axis=0)
print(models_df)


In [ ]:
!pip install tables

In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np

# Mount Google Drive
drive.mount('/content/drive')

# Define the base paths for each dataset
base_paths = [
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_2_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_1_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_0_ret_excess_fwd'
]

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Function to load all predictions from a specific HDF5 file
def load_predictions_from_hdf5(results_path):
    all_predictions = {}
    with pd.HDFStore(str(results_path / 'all_predictions.h5'), mode='r') as store:
        keys = store.keys()
        for key in keys:
            all_predictions[key] = store[key]
    return all_predictions

# Load predictions
def load_all_predictions(base_paths):
    predictions = {}
    for base_path_str in base_paths:
        base_path = Path(base_path_str)
        results_path = base_path / 'results' / 'asset_pricing'
        model_name = base_path.parts[-1]
        all_predictions = load_predictions_from_hdf5(results_path)
        predictions[model_name] = all_predictions
    return predictions


predictions = load_all_predictions(base_paths)


# Evaluate R^2 for xt
r2_values = {}
for model_name, predictions in tqdm(all_predictions.items()):
    for key, df_pred in predictions.items():

        df_pred = df_pred.sort_index()
        # Inner merge on MultiIndex
        print(f'DF pred {df_pred}')
        merged_df = pd.merge(df_pred, df, left_index=True, right_index=True, how='inner')

        columns_to_select = characteristics + ["y_pred"]
        characteristics_data_pred = merged_df[columns_to_select]

        columns_to_select = characteristics + ["y_true"]
        characteristics_data_true = merged_df[columns_to_select]

        # Example usage
        xt_true = calculate_xt_all_dates_true(characteristics_data_true)

        xt_pred = calculate_xt_all_dates_pred(characteristics_data_pred)

        xt_r2 = calculate_r2(xt_true, xt_pred)

        print(f"$R^2$ for $x_t$ for {model_name} for {key}:{xt_r2}")

# # Convert R^2 values to DataFrame for better visualization
# r2_df = pd.DataFrame(list(r2_values.items()), columns=['Model_Key', 'R^2'])
# print(r2_df)


# # Evaluate R^2 for xt
# r2_values = {}
# for model_name, predictions in tqdm(all_predictions.items()):
#     for key, df in predictions.items():
#         # Prepare characteristics data with y_pred
#         characteristics_data_pred = merged_df_padded[columns_to_select].drop(columns=['ret_excess']).join(df[['y_pred']], how='inner').rename(columns={"y_pred": "y_true"})

#         # Calculate xt for all dates with y_pred
#         xt_pred_df = calculate_xt_all_dates(characteristics_data_pred, y_column='y_true')

#         # Align indices to ensure consistent dates and stocks
#         common_index = xt_true_df.index.intersection(xt_pred_df.index)
#         xt_true_df_aligned = xt_true_df.loc[common_index]
#         xt_pred_df_aligned = xt_pred_df.loc[common_index]

#         # Calculate R^2 between xt_true and xt_pred
#         r2 = calculate_r2_xt(xt_true_df_aligned, xt_pred_df_aligned)
#         r2_values[f"{model_name}_{key}"] = r2

# # Convert R^2 values to DataFrame for better visualization
# r2_df = pd.DataFrame(list(r2_values.items()), columns=['Model_Key', 'R^2'])
# print(r2_df)


In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np

# Mount Google Drive
drive.mount('/content/drive')

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3_ret_excess_fwd')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Load a specific prediction from the HDF5 file
key_to_load = '/n_factors_1.0_l1_reg_1e-09'  # Replace with the key of the DataFrame you want to load
with pd.HDFStore(str(results_path / 'all_predictions.h5'), mode='r') as store:
    specific_prediction = store[key_to_load]

# Ensure 'df' has a MultiIndex ['date', 'permno']
# Assuming 'df' is the DataFrame containing the true values and the same characteristics
# specific_prediction = specific_prediction.set_index(['date', 'permno'])
# df = df.set_index(['date', 'permno'])  # Ensure the true DataFrame is also indexed the same way

# Sort the MultiIndex
specific_prediction = specific_prediction.sort_index()
df = df.sort_index()

# Inner merge on MultiIndex
merged_df = pd.merge(specific_prediction, df, left_index=True, right_index=True, how='inner')

# Function to calculate x_t for a given month using Ridge Regression for predicted values
def calculate_xt_y_pred(df, date, alpha=0):
    df_month = df.loc[date]
    r_t = df_month['y_pred'].values
    Z_t_minus_1 = df_month.drop(columns=['y_pred']).values
    Z_t_minus_1_T = Z_t_minus_1.T
    Z_t_minus_1_cov = Z_t_minus_1_T @ Z_t_minus_1
    Z_t_minus_1_cov_reg = Z_t_minus_1_cov + alpha * np.identity(Z_t_minus_1_cov.shape[0])
    Z_t_minus_1_cov_inv = np.linalg.inv(Z_t_minus_1_cov_reg)
    x_t = Z_t_minus_1_cov_inv @ Z_t_minus_1_T @ r_t
    return x_t, df_month.drop(columns=['y_pred']).columns

def calculate_xt_all_dates_pred(df, alpha=1e-6):
    xt_results = {}
    characteristic_columns = None
    unique_dates = df.index.get_level_values('date').unique()
    for date in unique_dates:
        x_t, characteristic_columns = calculate_xt_y_pred(df, date, alpha)
        xt_results[date] = x_t
    xt_df = pd.DataFrame.from_dict(xt_results, orient='index')
    if characteristic_columns is not None:
        xt_df.columns = characteristic_columns
    xt_df.index.name = 'date'
    return xt_df

# Function to calculate x_t for a given month using Ridge Regression for true values
def calculate_xt_y_true(df, date, alpha=0):
    df_month = df.loc[date]
    r_t = df_month['y_true'].values
    Z_t_minus_1 = df_month.drop(columns=['y_true']).values
    Z_t_minus_1_T = Z_t_minus_1.T
    Z_t_minus_1_cov = Z_t_minus_1_T @ Z_t_minus_1
    Z_t_minus_1_cov_reg = Z_t_minus_1_cov + alpha * np.identity(Z_t_minus_1_cov.shape[0])
    Z_t_minus_1_cov_inv = np.linalg.inv(Z_t_minus_1_cov_reg)
    x_t = Z_t_minus_1_cov_inv @ Z_t_minus_1_T @ r_t
    return x_t, df_month.drop(columns=['y_true']).columns

def calculate_xt_all_dates_true(df, alpha=1e-6):
    xt_results = {}
    characteristic_columns = None
    unique_dates = df.index.get_level_values('date').unique()
    for date in unique_dates:
        x_t, characteristic_columns = calculate_xt_y_true(df, date, alpha)
        xt_results[date] = x_t
    xt_df = pd.DataFrame.from_dict(xt_results, orient='index')
    if characteristic_columns is not None:
        xt_df.columns = characteristic_columns
    xt_df.index.name = 'date'
    return xt_df

# Calculate x_t for true and predicted values
columns_to_select_pred = [col for col in merged_df.columns if 'y_pred' in col or col.startswith('characteristic')]
characteristics_data_pred = merged_df[columns_to_select_pred]

columns_to_select_true = [col for col in merged_df.columns if 'y_true' in col or col.startswith('characteristic')]
characteristics_data_true = merged_df[columns_to_select_true]

xt_true = calculate_xt_all_dates_true(characteristics_data_true)
xt_pred = calculate_xt_all_dates_pred(characteristics_data_pred)

# Step 1: Calculate the residuals
residuals = xt_true - xt_pred

# Step 2: Compute the sum of squared residuals (SSR)
ssr = np.sum(np.square(residuals.values))

# Step 3: Compute the total sum of squares (TSS)
tss = np.sum(np.square(xt_true.values - xt_true.mean().values))

# Step 4: Calculate R^2
r_squared = 1 - (ssr / tss)

print("R^2 for x_t:", r_squared)


In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np

# Mount Google Drive
drive.mount('/content/drive')

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3_ret_excess_fwd')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Load all predictions from the HDF5 file
all_predictions = {}
with pd.HDFStore(str(results_path / 'all_predictions.h5'), mode='r') as store:
    keys = store.keys()
    for key in keys:
        all_predictions[key] = store[key]

# Define function to merge and calculate xt for predicted values
def merge_and_calculate_xt(pred_df, true_df):
    pred_df = pred_df.sort_index()
    true_df = true_df.sort_index()
    merged_df = pd.merge(pred_df, true_df, left_index=True, right_index=True, how='inner')

    columns_to_select_pred = [col for col in merged_df.columns if 'y_pred' in col or col.startswith('characteristic')]
    characteristics_data_pred = merged_df[columns_to_select_pred]

    columns_to_select_true = [col for col in merged_df.columns if 'y_true' in col or col.startswith('characteristic')]
    characteristics_data_true = merged_df[columns_to_select_true]

    xt_true = calculate_xt_all_dates_true(characteristics_data_true)
    xt_pred = calculate_xt_all_dates_pred(characteristics_data_pred)

    return xt_true, xt_pred

# Function to calculate x_t for a given month using Ridge Regression for predicted values
def calculate_xt_y_pred(df, date, alpha=0):
    df_month = df.loc[date]
    r_t = df_month['y_pred'].values
    Z_t_minus_1 = df_month.drop(columns=['y_pred']).values
    Z_t_minus_1_T = Z_t_minus_1.T
    Z_t_minus_1_cov = Z_t_minus_1_T @ Z_t_minus_1
    Z_t_minus_1_cov_reg = Z_t_minus_1_cov + alpha * np.identity(Z_t_minus_1_cov.shape[0])
    Z_t_minus_1_cov_inv = np.linalg.inv(Z_t_minus_1_cov_reg)
    x_t = Z_t_minus_1_cov_inv @ Z_t_minus_1_T @ r_t
    return x_t, df_month.drop(columns=['y_pred']).columns

# Function to calculate x_t for all unique dates in the dataset
def calculate_xt_all_dates_pred(df, alpha=1e-6):
    xt_results = {}
    characteristic_columns = None
    unique_dates = df.index.get_level_values('date').unique()
    for date in unique_dates:
        x_t, characteristic_columns = calculate_xt_y_pred(df, date, alpha)
        xt_results[date] = x_t
    xt_df = pd.DataFrame.from_dict(xt_results, orient='index')
    if characteristic_columns is not None:
        xt_df.columns = characteristic_columns
    xt_df.index.name = 'date'
    return xt_df

# Function to calculate x_t for a given month using Ridge Regression for true values
def calculate_xt_y_true(df, date, alpha=0):
    df_month = df.loc[date]
    r_t = df_month['y_true'].values
    Z_t_minus_1 = df_month.drop(columns=['y_true']).values
    Z_t_minus_1_T = Z_t_minus_1.T
    Z_t_minus_1_cov = Z_t_minus_1_T @ Z_t_minus_1
    Z_t_minus_1_cov_reg = Z_t_minus_1_cov + alpha * np.identity(Z_t_minus_1_cov.shape[0])
    Z_t_minus_1_cov_inv = np.linalg.inv(Z_t_minus_1_cov_reg)
    x_t = Z_t_minus_1_cov_inv @ Z_t_minus_1_T @ r_t
    return x_t, df_month.drop(columns=['y_true']).columns

# Function to calculate x_t for all unique dates in the dataset
def calculate_xt_all_dates_true(df, alpha=1e-6):
    xt_results = {}
    characteristic_columns = None
    unique_dates = df.index.get_level_values('date').unique()
    for date in unique_dates:
        x_t, characteristic_columns = calculate_xt_y_true(df, date, alpha)
        xt_results[date] = x_t
    xt_df = pd.DataFrame.from_dict(xt_results, orient='index')
    if characteristic_columns is not None:
        xt_df.columns = characteristic_columns
    xt_df.index.name = 'date'
    return xt_df

# Assuming df contains the true values DataFrame and is indexed correctly
# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3_ret_excess_fwd')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Load the true values DataFrame (assuming it is named 'true_values.h5' in the same folder)
true_values_path = results_path / 'all_predictions.h5'
with pd.HDFStore(str(true_values_path), mode='r') as store:
    df = store['true_values']  # Assuming the true values DataFrame is stored with the key 'true_values'

# Calculate R^2 for each prediction DataFrame
r_squared_results = {}
for key, pred_df in all_predictions.items():
    xt_true, xt_pred = merge_and_calculate_xt(pred_df, df)

    # Calculate residuals
    residuals = xt_true - xt_pred

    # Calculate SSR and TSS
    ssr = np.sum(np.square(residuals.values))
    tss = np.sum(np.square(xt_true.values - xt_true.mean().values))

    # Calculate R^2
    r_squared = 1 - (ssr / tss)
    r_squared_results[key] = r_squared

# Display R^2 results
for key, r_squared in r_squared_results.items():
    print(f"R^2 for {key}: {r_squared}")


In [ ]:
print(xt_pred.shape,xt_true.shape)


In [ ]:
print(characteristics_data_true.columns)

print(merged_df.head())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
from pathlib import Path
from google.colab import drive
from scipy.linalg import solve
from tqdm import tqdm

# Mount Google Drive
drive.mount('/content/drive')

# Define the base paths for each dataset
base_paths = [
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_2_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_1_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_0_ret_excess_fwd'
]

# Function to load all predictions from a specific HDF5 file
def load_predictions_from_hdf5(results_path):
    all_predictions = {}
    with pd.HDFStore(str(results_path / 'all_predictions.h5'), mode='r') as store:
        keys = store.keys()
        for key in keys:
            all_predictions[key] = store[key]
    return all_predictions

# Function to filter out columns with -2 values
def filter_columns(df):
    return df.loc[:, (df != -2).any(axis=0)]

# Function to calculate x_t for a given month using Ridge Regression
def calculate_xt(df, date, y_column='y_true', alpha=0):
    # Filter DataFrame for the given date
    df_month = df.loc[date]

    # Separate returns and characteristics
    r_t = df_month[y_column].values
    Z_t_minus_1 = df_month.drop(columns=[y_column])

    # Filter out columns with all -2 values
    Z_t_minus_1 = filter_columns(Z_t_minus_1)

    # Calculate (Z_{t-1}'Z_{t-1})^{-1}Z_{t-1}r_t using Ridge Regression
    Z_t_minus_1_T = Z_t_minus_1.T.values
    Z_t_minus_1_cov = Z_t_minus_1_T @ Z_t_minus_1.values
    Z_t_minus_1_cov_reg = Z_t_minus_1_cov + alpha * np.identity(Z_t_minus_1_cov.shape[0])

    # Solve for x_t
    x_t = solve(Z_t_minus_1_cov_reg, Z_t_minus_1_T @ r_t, assume_a='pos')

    return x_t, Z_t_minus_1.columns

# Calculate x_t for all unique dates in the dataset
def calculate_xt_all_dates(df, y_column='y_true', alpha=1e-6):
    xt_results = {}
    characteristic_columns = None

    unique_dates = df.index.get_level_values('date').unique()
    for date in unique_dates:
        x_t, characteristic_columns = calculate_xt(df, date, y_column=y_column, alpha=alpha)
        xt_results[date] = x_t

    # Convert dictionary to DataFrame
    xt_df = pd.DataFrame.from_dict(xt_results, orient='index')

    # Assign original characteristic names as column names
    if characteristic_columns is not None:
        xt_df.columns = characteristic_columns

    # Preserve the index name
    xt_df.index.name = 'date'

    return xt_df

# Load predictions
def load_all_predictions(base_paths):
    predictions = {}
    for base_path_str in base_paths:
        base_path = Path(base_path_str)
        results_path = base_path / 'results' / 'asset_pricing'
        model_name = base_path.parts[-1]
        all_predictions = load_predictions_from_hdf5(results_path)
        predictions[model_name] = all_predictions
    return predictions

# Calculate R^2 for xt values
def calculate_r2_xt(xt_true_df, xt_pred_df):
    y_true_flat = xt_true_df.values.flatten()
    y_pred_flat = xt_pred_df.values.flatten()
    return r2_score(y_true_flat, y_pred_flat)

# Example usage
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/beta.parquet'
merged_df_padded = pd.read_parquet(file_path)
merged_df_padded.index = merged_df_padded.index.set_levels(pd.to_datetime(merged_df_padded.index.levels[0]), level=0)
merged_df_padded = merged_df_padded.loc[merged_df_padded.index.get_level_values('date') >= '1960-01-01']

# Load all predictions
all_predictions = load_all_predictions(base_paths)

# Prepare merged data
characteristics = [col for col in merged_df_padded.columns if 'characteristic' in col]
columns_to_select = characteristics + ["ret_excess"]
characteristics_data = merged_df_padded[columns_to_select]

# Calculate xt for all dates with y_true
characteristics_data_true = characteristics_data.rename(columns={"ret_excess": "y_true"})
xt_true_df = calculate_xt_all_dates(characteristics_data_true, y_column='y_true')

# Evaluate R^2 for xt
r2_values = {}
for model_name, predictions in tqdm(all_predictions.items()):
    for key, df in predictions.items():
        # Prepare characteristics data with y_pred
        characteristics_data_pred = merged_df_padded[columns_to_select].drop(columns=['ret_excess']).join(df[['y_pred']], how='inner').rename(columns={"y_pred": "y_true"})

        # Calculate xt for all dates with y_pred
        xt_pred_df = calculate_xt_all_dates(characteristics_data_pred, y_column='y_true')

        # Align indices to ensure consistent dates and stocks
        common_index = xt_true_df.index.intersection(xt_pred_df.index)
        xt_true_df_aligned = xt_true_df.loc[common_index]
        xt_pred_df_aligned = xt_pred_df.loc[common_index]

        # Calculate R^2 between xt_true and xt_pred
        r2 = calculate_r2_xt(xt_true_df_aligned, xt_pred_df_aligned)
        r2_values[f"{model_name}_{key}"] = r2

# Convert R^2 values to DataFrame for better visualization
r2_df = pd.DataFrame(list(r2_values.items()), columns=['Model_Key', 'R^2'])
print(r2_df)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
from pathlib import Path
from google.colab import drive
from scipy.linalg import solve
from tqdm import tqdm

# Mount Google Drive
drive.mount('/content/drive')

# Define the base paths for each dataset
base_paths = [
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_2_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_1_ret_excess_fwd',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_0_ret_excess_fwd'
]

# Function to load all predictions from a specific HDF5 file
def load_predictions_from_hdf5(results_path):
    all_predictions = {}
    with pd.HDFStore(str(results_path / 'all_predictions.h5'), mode='r') as store:
        keys = store.keys()
        for key in keys:
            all_predictions[key] = store[key]
    return all_predictions

# Function to filter out columns with -2 values
def filter_columns(df):
    return df.loc[:, (df != -2).any(axis=0)]

# Function to calculate x_t for a given month using Ridge Regression
def calculate_xt(df, date, y_column='y_true', alpha=0):
    # Filter DataFrame for the given date
    df_month = df.loc[date]

    # Separate returns and characteristics
    r_t = df_month[y_column].values
    Z_t_minus_1 = df_month.drop(columns=[y_column])

    # Filter out columns with all -2 values
    Z_t_minus_1 = filter_columns(Z_t_minus_1)

    # Calculate (Z_{t-1}'Z_{t-1})^{-1}Z_{t-1}r_t using Ridge Regression
    Z_t_minus_1_T = Z_t_minus_1.T.values
    Z_t_minus_1_cov = Z_t_minus_1_T @ Z_t_minus_1.values
    Z_t_minus_1_cov_reg = Z_t_minus_1_cov + alpha * np.identity(Z_t_minus_1_cov.shape[0])

    # Solve for x_t
    x_t = solve(Z_t_minus_1_cov_reg, Z_t_minus_1_T @ r_t, assume_a='pos')

    return x_t, Z_t_minus_1.columns

# Calculate x_t for all unique dates in the dataset
def calculate_xt_all_dates(df, y_column='y_true', alpha=1e-6):
    xt_results = {}
    characteristic_columns = None

    unique_dates = df.index.get_level_values('date').unique()
    for date in unique_dates:
        x_t, characteristic_columns = calculate_xt(df, date, y_column=y_column, alpha=alpha)
        xt_results[date] = x_t

    # Convert dictionary to DataFrame
    xt_df = pd.DataFrame.from_dict(xt_results, orient='index')

    # Assign original characteristic names as column names
    if characteristic_columns is not None:
        xt_df.columns = characteristic_columns

    # Preserve the index name
    xt_df.index.name = 'date'

    return xt_df

# Load predictions
def load_all_predictions(base_paths):
    predictions = {}
    for base_path_str in base_paths:
        base_path = Path(base_path_str)
        results_path = base_path / 'results' / 'asset_pricing'
        model_name = base_path.parts[-1]
        all_predictions = load_predictions_from_hdf5(results_path)
        predictions[model_name] = all_predictions
    return predictions

# Calculate R^2 for xt values
def calculate_r2_xt(xt_true_df, xt_pred_df):
    y_true_flat = xt_true_df.values.flatten()
    y_pred_flat = xt_pred_df.values.flatten()
    return r2_score(y_true_flat, y_pred_flat)

# Example usage
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/beta.parquet'
merged_df_padded = pd.read_parquet(file_path)
merged_df_padded.index = merged_df_padded.index.set_levels(pd.to_datetime(merged_df_padded.index.levels[0]), level=0)
merged_df_padded = merged_df_padded.loc[merged_df_padded.index.get_level_values('date') >= '1960-01-01']

# Load all predictions
all_predictions = load_all_predictions(base_paths)

# Prepare merged data
characteristics = [col for col in merged_df_padded.columns if 'characteristic' in col]
columns_to_select = characteristics + ["ret_excess"]
characteristics_data = merged_df_padded[columns_to_select]

# Calculate xt for all dates with y_true
characteristics_data_true = characteristics_data.rename(columns={"ret_excess": "y_true"})
xt_true_df = calculate_xt_all_dates(characteristics_data_true, y_column='y_true')

# Evaluate R^2 for xt
r2_values = []
for model_name, predictions in tqdm(all_predictions.items()):
    for key, df in predictions.items():
        # Prepare characteristics data with y_pred
        characteristics_data_pred = merged_df_padded[columns_to_select].drop(columns=['ret_excess']).join(df[['y_pred']], how='inner').rename(columns={"y_pred": "y_true"})

        # Calculate xt for all dates with y_pred
        xt_pred_df = calculate_xt_all_dates(characteristics_data_pred, y_column='y_true')

        # Align indices to ensure consistent dates and stocks
        common_index = xt_true_df.index.intersection(xt_pred_df.index)
        xt_true_df_aligned = xt_true_df.loc[common_index]
        xt_pred_df_aligned = xt_pred_df.loc[common_index]

        # Calculate R^2 between xt_true and xt_pred
        r2 = calculate_r2_xt(xt_true_df_aligned, xt_pred_df_aligned)
        n_factors = key.split('_')[-1]
        r2_values.append([model_name, n_factors, r2])
# ... (rest of the code remains the same)

# Convert R^2 values to DataFrame for better visualization
r2_df = pd.DataFrame(r2_values, columns=['Model', 'n_factors', 'R^2'])

# Combine Model and n_factors to ensure uniqueness
r2_df['Model_Factors'] = r2_df['Model'] + '_' + r2_df['n_factors'].astype(str)

# Pivot the DataFrame using the unique 'Model_Factors' column as the index
r2_pivot = r2_df.pivot(index='Model_Factors', columns='n_factors', values='R^2') # Use the combined column for unique index

# Format the values to two decimal places
r2_pivot = r2_pivot.applymap(lambda x: f"{x:.2f}")

# Generate LaTeX table
latex_table = r2_pivot.to_latex()

print(latex_table)

In [ ]:
from sklearn.metrics import r2_score
import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import drive
from tqdm import tqdm
import concurrent.futures
from scipy.linalg import solve

# Mount Google Drive
drive.mount('/content/drive')

# Define the base path for a single dataset for testing
base_path = '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3_ret_excess_fwd'

# Function to load all predictions from a specific HDF5 file
def load_predictions_from_hdf5(results_path):
    all_predictions = {}
    with pd.HDFStore(str(results_path / 'all_predictions.h5'), mode='r') as store:
        keys = store.keys()
        # Load only the first key for testing
        key = keys[0]
        all_predictions[key] = store[key]
    return all_predictions

# Function to calculate x_t for a given month using Ridge Regression
def calculate_xt(df, date, y_column='y_true', alpha=0):
    # Filter DataFrame for the given date
    df_month = df.loc[date]

    # Separate returns and characteristics
    r_t = df_month[y_column].values
    characteristic_columns = df_month.drop(columns=['y_true', 'y_pred']).columns
    Z_t_minus_1 = df_month.drop(columns=['y_true', 'y_pred']).values

    # Calculate (Z_{t-1}'Z_{t-1})^{-1}Z_{t-1}r_t using Ridge Regression
    Z_t_minus_1_T = Z_t_minus_1.T
    Z_t_minus_1_cov = Z_t_minus_1_T @ Z_t_minus_1
    Z_t_minus_1_cov_reg = Z_t_minus_1_cov + alpha * np.identity(Z_t_minus_1_cov.shape[0])

    # Use scipy.solve for efficient solving of linear equations
    x_t = solve(Z_t_minus_1_cov_reg, Z_t_minus_1_T @ r_t, assume_a='pos')

    return x_t, characteristic_columns

# Calculate x_t for all unique dates in the dataset
def calculate_xt_all_dates(df, alpha=1e-6):
    xt_true_results = {}
    xt_pred_results = {}
    characteristic_columns = None

    unique_dates = df.index.get_level_values('date').unique()
    for date in unique_dates:
        x_t_true, characteristic_columns = calculate_xt(df, date, y_column='y_true', alpha=alpha)
        x_t_pred, _ = calculate_xt(df, date, y_column='y_pred', alpha=alpha)
        xt_true_results[date] = x_t_true
        xt_pred_results[date] = x_t_pred

    # Convert dictionary to DataFrame
    xt_true_df = pd.DataFrame.from_dict(xt_true_results, orient='index')
    xt_pred_df = pd.DataFrame.from_dict(xt_pred_results, orient='index')

    # Assign original characteristic names as column names
    if characteristic_columns is not None:
        xt_true_df.columns = characteristic_columns
        xt_pred_df.columns = characteristic_columns

    # Preserve the index name
    xt_true_df.index.name = 'date'
    xt_pred_df.index.name = 'date'

    return xt_true_df, xt_pred_df

def parallel_calculate_xt_all_dates(df, alpha=1e-6, n_jobs=4):
    unique_dates = df.index.get_level_values('date').unique()
    date_batches = np.array_split(unique_dates, n_jobs)

    xt_true_results = []
    xt_pred_results = []
    with concurrent.futures.ProcessPoolExecutor(max_workers=n_jobs) as executor:
        futures = {
            executor.submit(calculate_xt_all_dates, df.loc[batch], alpha): batch
            for batch in date_batches
        }

        for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc='Batches processed in calculate xt'):
            xt_true_df, xt_pred_df = future.result()
            xt_true_results.append(xt_true_df)
            xt_pred_results.append(xt_pred_df)

    # Combine all the results
    xt_true_df = pd.concat(xt_true_results)
    xt_pred_df = pd.concat(xt_pred_results)
    return xt_true_df, xt_pred_df

# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade"]
# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes
characteristics = [f'characteristic_{code}' for code in all_codes]

# Load characteristics data
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/beta.parquet'
merged_df_padded = pd.read_parquet(file_path)
merged_df_padded.index = merged_df_padded.index.set_levels(pd.to_datetime(merged_df_padded.index.levels[0]), level=0)
merged_df_padded = merged_df_padded.loc[merged_df_padded.index.get_level_values('date') >= '1960-01-01']

# Load predictions
results_path = Path(base_path) / 'results' / 'asset_pricing'
all_predictions = load_predictions_from_hdf5(results_path)

# Calculate R^2 for a single key
key = list(all_predictions.keys())[0]
print(f"Processing key: {key}")
df = all_predictions[key]

# Prepare the merged data
merged_df_padded = merged_df_padded.loc[df.index.get_level_values('date').unique()]

# Combine characteristics and predictions
combined_data = merged_df_padded.join(df, how='inner')

# Calculate x_t in parallel
xt_true_df, xt_pred_df = parallel_calculate_xt_all_dates(combined_data)
batch
if not xt_true_df.empty and not xt_pred_df.empty:
    y_true_flat = xt_true_df.values.flatten()
    y_pred_flat = xt_pred_df.values.flatten()
    r2 = r2_score(y_true_flat, y_pred_flat)
    print(f"R^2 for {key}: {r2:.4f}")
else:
    print(f"No valid data to calculate R^2 for {key}.")


In [ ]:
from sklearn.metrics import r2_score
import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import drive
from scipy.linalg import solve

# Mount Google Drive
drive.mount('/content/drive')

# Define the base path for datasets
base_path = '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3_ret_excess_fwd'

# Function to load all predictions from a specific HDF5 file
def load_predictions_from_hdf5(results_path):
    all_predictions = {}
    with pd.HDFStore(str(results_path / 'all_predictions.h5'), mode='r') as store:
        keys = store.keys()
        for key in keys:
            all_predictions[key] = store[key]
    return all_predictions

# Function to calculate x_t for a given month using Ridge Regression
def calculate_xt(df, date, y_column='y_true', alpha=0):
    # Filter DataFrame for the given date
    df_month = df.loc[date]

    # Separate returns and characteristics
    r_t = df_month[y_column].values
    characteristic_columns = df_month.drop(columns=['y_true', 'y_pred']).columns
    Z_t_minus_1 = df_month.drop(columns=['y_true', 'y_pred']).values

    # Calculate (Z_{t-1}'Z_{t-1})^{-1}Z_{t-1}r_t using Ridge Regression
    Z_t_minus_1_T = Z_t_minus_1.T
    Z_t_minus_1_cov = Z_t_minus_1_T @ Z_t_minus_1
    Z_t_minus_1_cov_reg = Z_t_minus_1_cov + alpha * np.identity(Z_t_minus_1_cov.shape[0])

    # Use scipy.solve for efficient solving of linear equations
    x_t = solve(Z_t_minus_1_cov_reg, Z_t_minus_1_T @ r_t, assume_a='pos')

    return x_t, characteristic_columns

# Calculate x_t for all unique dates in the dataset
def calculate_xt_all_dates(df, alpha=1e-6):
    xt_true_results = {}
    xt_pred_results = {}
    characteristic_columns = None

    unique_dates = df.index.get_level_values('date').unique()
    for date in unique_dates:
        x_t_true, characteristic_columns = calculate_xt(df, date, y_column='y_true', alpha=alpha)
        x_t_pred, _ = calculate_xt(df, date, y_column='y_pred', alpha=alpha)
        xt_true_results[date] = x_t_true
        xt_pred_results[date] = x_t_pred

    # Convert dictionary to DataFrame
    xt_true_df = pd.DataFrame.from_dict(xt_true_results, orient='index')
    xt_pred_df = pd.DataFrame.from_dict(xt_pred_results, orient='index')

    # Assign original characteristic names as column names
    if characteristic_columns is not None:
        xt_true_df.columns = characteristic_columns
        xt_pred_df.columns = characteristic_columns

    # Preserve the index name
    xt_true_df.index.name = 'date'
    xt_pred_df.index.name = 'date'

    return xt_true_df, xt_pred_df

# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade"]
# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes
characteristics = [f'characteristic_{code}' for code in all_codes]

# Load characteristics data
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/beta.parquet'
merged_df_padded = pd.read_parquet(file_path)
merged_df_padded.index = merged_df_padded.index.set_levels(pd.to_datetime(merged_df_padded.index.levels[0]), level=0)
merged_df_padded = merged_df_padded.loc[merged_df_padded.index.get_level_values('date') >= '1960-01-01']

# Load predictions
results_path = Path(base_path) / 'results' / 'asset_pricing'
all_predictions = load_predictions_from_hdf5(results_path)

# Prepare the merged data
merged_df_padded = merged_df_padded.loc[all_predictions[list(all_predictions.keys())[0]].index.get_level_values('date').unique()]

# Combine characteristics and predictions
combined_data = merged_df_padded.join(all_predictions[list(all_predictions.keys())[0]], how='inner')

# Iterate through all prediction keys
r2_scores = {}
for key, df in all_predictions.items():
    print(f"Processing key: {key}")
    combined_data = merged_df_padded.join(df, how='inner')

    # Calculate x_t
    xt_true_df, xt_pred_df = calculate_xt_all_dates(combined_data)

    if not xt_true_df.empty and not xt_pred_df.empty:
        y_true_flat = xt_true_df.values.flatten()
        y_pred_flat = xt_pred_df.values.flatten()
        r2 = r2_score(y_true_flat, y_pred_flat)
        r2_scores[key] = r2
        print(f"R^2 for {key}: {r2:.4f}")
    else:
        print(f"No valid data to calculate R^2 for {key}.")

# Print R^2 scores
print("R^2 Scores for all prediction keys:")
for key, r2 in r2_scores.items():
    print(f"{key}: {r2:.4f}")


In [ ]:
from sklearn.metrics import r2_score
import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import drive
from tqdm import tqdm

# Mount Google Drive
drive.mount('/content/drive')

# Define the base path for a single dataset for testing
base_path = '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3_ret_excess_fwd'

# Function to load all predictions from a specific HDF5 file
def load_predictions_from_hdf5(results_path):
    all_predictions = {}
    with pd.HDFStore(str(results_path / 'all_predictions.h5'), mode='r') as store:
        keys = store.keys()
        # Load only the first key for testing
        key = keys[0]
        all_predictions[key] = store[key]
    return all_predictions

def calculate_xt(beta_with_noise, result, columns_to_select):
    xt_true_list = []
    xt_pred_list = []

    beta_with_noise = beta_with_noise.sort_index(level=0)
    result = result.sort_index(level=0)

    common_dates = result.index.get_level_values(0).unique()
    beta_with_noise = beta_with_noise.loc[common_dates]

    for date in tqdm(common_dates, desc='Dates processed in calculate xt'):
        try:
            Z_t_1 = beta_with_noise.loc[date, columns_to_select].replace(-2, np.nan).dropna()

            # Get common permnos for the current date
            common_permnos = Z_t_1.index.intersection(result.loc[date].index)

            if not common_permnos.empty:
                Z_t_1 = Z_t_1.loc[common_permnos]
                r_t_true = result.loc[(date, common_permnos), 'y_true'].dropna()
                r_t_pred = result.loc[(date, common_permnos), 'y_pred'].dropna()

                if not Z_t_1.empty and not r_t_true.empty and len(Z_t_1) == len(r_t_true):
                    Z_t_1 = Z_t_1.astype(np.float32).values
                    r_t_true = r_t_true.astype(np.float32).values
                    r_t_pred = r_t_pred.astype(np.float32).values

                    Z_t_1_T = Z_t_1.T
                    Z_t_1_Z_t_1_inv = np.linalg.inv(Z_t_1_T @ Z_t_1)

                    x_t_true = Z_t_1_Z_t_1_inv @ Z_t_1_T @ r_t_true
                    xt_true_list.append(pd.DataFrame(x_t_true.reshape(1, -1), index=[date], columns=[f'x_{i+1}' for i in range(x_t_true.shape[0])]))

                    x_t_pred = Z_t_1_Z_t_1_inv @ Z_t_1_T @ r_t_pred
                    xt_pred_list.append(pd.DataFrame(x_t_pred.reshape(1, -1), index=[date], columns=[f'x_{i+1}' for i in range(x_t_pred.shape[0])]))
                else:
                    print(f"Skipping date {date} due to mismatched dimensions or empty data.")
            else:
                print(f"No common permnos for date {date}, skipping.")
        except Exception as e:
            print(f"Error processing date {date}: {e}")
            continue

    xt_df_true = pd.concat(xt_true_list)
    xt_df_pred = pd.concat(xt_pred_list)

    return xt_df_true, xt_df_pred

# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade"]
# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes
characteristics = [f'characteristic_{code}' for code in all_codes]

# Select the required columns including the noise characteristics
columns_to_select = characteristics

# Load characteristics data
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/beta.parquet'
merged_df_padded = pd.read_parquet(file_path)
merged_df_padded.index = merged_df_padded.index.set_levels(pd.to_datetime(merged_df_padded.index.levels[0]), level=0)
merged_df_padded = merged_df_padded.loc[merged_df_padded.index.get_level_values('date') >= '1960-01-01']

# Replace all -2 values with NaN
merged_df_padded.replace(-2, np.nan, inplace=True)

# Load predictions
results_path = Path(base_path) / 'results' / 'asset_pricing'
all_predictions = load_predictions_from_hdf5(results_path)

# Calculate R^2 for a single key
key = list(all_predictions.keys())[0]
print(f"Processing key: {key}")
df = all_predictions[key]
xt_df_true, xt_df_pred = calculate_xt(merged_df_padded, df, columns_to_select)

if not xt_df_true.empty and not xt_df_pred.empty:
    y_true_flat = xt_df_true.values.flatten()
    y_pred_flat = xt_df_pred.values.flatten()
    r2 = r2_score(y_true_flat, y_pred_flat)
    print(f"R^2 for {key}: {r2:.4f}")
else:
    print(f"No valid data to calculate R^2 for {key}.")


In [ ]:
df

**Total $R^2$ for $r_t$**